# Wo hört Whisper hin? – Zeitliche und spektrale Lokalisierung der Evidenz für einzelne Wörter

**Forschungsfrage:** Welche Zeitabschnitte und akustischen Merkmale des Eingangssignals sind für die Transkription einzelner Wörter durch Whisper entscheidend – und wie gut lokalisieren Input-Attribution und Attention-Mechanismen diese Evidenz im Vergleich zu den tatsächlichen Wortgrenzen?

| Teilfrage | Methode | Abschnitt |
|---|---|---|
| TF1 Zeitliche Lokalisierung | Integrated Gradients, Gradient×Δ, Zeit-Occlusion vs. MFA-Wortgrenzen | 8 |
| TF2 Spektrale Merkmale | IG-Profil über Mel-Bänder, Frequenzband-Occlusion; Funktions- vs. Inhaltswörter | 9 |
| TF3 Cross-Attention | Pointing Game / Masse-im-Wort je Decoder-Kopf, Vergleich mit OpenAIs *alignment heads* | 10 |
| TF4 Encoder-Self-Attention | Kopf-Statistiken (Lokalität, Distanz, Padding-Sink, Wortbindung) je Schicht | 11 |
| TF5 Faithfulness | Deletion-Test je Methode, Ablation einzelner Köpfe (Δ log p, WER) | 12 |
| Skalierung | whisper-tiny vs. whisper-base | 13 |
| Methodencheck | Konvergenz von Integrated Gradients | 7c |

Dokumentation, Hypothesen und Quellen: siehe [`README.md`](README.md). Laufzeitumgebung: CUDA-GPU erforderlich (getestet mit GTX 1660 SUPER, 6 GB); Installation, Probelauf und Ergebnis-Cache sind in Abschnitt 0 beschrieben.

### Kurzantwort

Ausgewertet wurden 100 LibriSpeech-Äußerungen (ca. 1750 korrekt erkannte Wörter je Modell), für Deletion und Ablation nur 25. Die Zahlen stehen in den Abschnitten 8–13, die Hypothesen werden in Abschnitt 14 beantwortet.

1. **Whisper hört im Wortsegment hin (H1 bestätigt).** Alle Methoden liegen weit über Zufall (`mass_in` > `chance`, p < 1e-287). Der Kontext vor und nach dem Wort ist bei IG, Grad×Δ und Occlusion etwa gleich groß (je ca. 0,09–0,11). Occlusion lokalisiert am schärfsten (`mass_in` 0,57–0,64 gegen 0,21–0,23 bei IG).
2. **Die IG-Ergebnisse sind nicht konvergiert.** Der Completeness-Fehler liegt im Median bei 32 % (tiny) bzw. 28 % (base) statt unter 5 %. IG-Zahlen sind deshalb nur als grobe Näherung zu lesen. Occlusion, Cross-Attention und Ablation sind davon nicht betroffen.
3. **Nur wenige Cross-Attention-Köpfe lokalisieren (H2 teilweise bestätigt).** Die `alignment_heads` sind im Mittel deutlich besser als der Durchschnitt (Pointing 0,58/0,73 gegen 0,31/0,22), aber nicht exakt die besten Köpfe. Der Mittelwert über alle Köpfe ist als Lokalisierer unbrauchbar (Pointing ≈ Zufall).
4. **Lokal → global über die Schichten (H3) ist statistisch nicht bestätigt.** Spearman über die Schichten: p = 0,30 (tiny) bzw. 0,41 (base). Nur der Vergleich der Schicht 0 mit der letzten Schicht zeigt beschreibend weniger lokale Attention (`local` 0,41 → 0,16 bzw. 0,35 → 0,12).
5. **Attention ist nicht schwächer faithful als Attribution (H4a verworfen), aber Lokalisierung sagt Wirkung nicht voraus (H4b bestätigt).** Die Cross-Attention der Alignment-Köpfe ist im Deletion-Test so treu wie IG. Die kausal wichtigsten Köpfe lokalisieren aber schlecht (Spearman ρ = −0,56 / −0,54).
6. **Alignment-Köpfe sind nicht generell kausal wichtiger als Zufallsköpfe.** Bei tiny ja, bei base schaden einzelne Zufallsmengen mehr. Der WER über 100 % in der Ablation ist vermutlich Halluzination bzw. Wiederholung (siehe Abschnitt 12).
7. **Spektral zählt vor allem der Bereich 410–745 Hz** (erster Formant).

## 0 Installation und Laufmodi

```
python -m venv .venv
.venv\Scripts\python -m pip install -r requirements.txt
```

`requirements.txt` pinnt die Versionen, mit denen die Ergebnisse erzeugt wurden (u. a. PyTorch 2.11.0 mit CUDA 12.8, Transformers 5.23.0, Datasets 5.2.0). Danach im Notebook den Kernel der `.venv` wählen. Die Setup-Zelle bricht ab, wenn CUDA nicht verfügbar ist, statt unbemerkt auf der CPU zu rechnen.

* **Ergebnis-Cache:** Die Hauptschleife (Abschnitt 7) speichert alle Ergebnisse zusammen mit der `CFG` in `results/cache_res.pkl`. Mit `LOAD_CACHED = True` (Standard) wird dieser Cache geladen, sofern die `CFG` identisch ist. Alle Auswertungszellen laufen dann in Sekunden, ohne die Modelle neu zu rechnen. Bei geänderter `CFG` wird automatisch neu gerechnet.
* **Probelauf:** Mit der Umgebungsvariable `WHISPER_XAI_QUICK=1` läuft eine Mini-Konfiguration (10 Äußerungen, 3 für Deletion/Ablation, weniger IG-Schritte). Sie schreibt in einen eigenen Ordner `results_quick/` und überschreibt die Hauptergebnisse nicht.
* **Laufzeit:** Der vollständige Lauf dauert auf einer GTX 1660 SUPER ca. 15 Stunden (tiny ca. 4,5 h, base ca. 10 h). Den größten Anteil hat Integrated Gradients mit 256 Stützstellen (ca. 150 s bzw. 320 s pro Äußerung).
* **Speicher:** `batch` und `ig_batch` in `CFG` sind für 6 GB VRAM gewählt. Bei Speicherfehlern beide auf 1 setzen; die Ergebnisse ändern sich dadurch nicht.

In [ ]:
import io, os, re, math, json, zlib, random, pickle, difflib, warnings
import logging
warnings.filterwarnings("ignore")                    # vor den Imports, damit auch Import-Warnungen (tqdm) entfallen
logging.getLogger("torch.distributed.elastic").setLevel(logging.ERROR)   # Windows-Hinweis von torch
from collections import defaultdict
from contextlib import contextmanager, nullcontext
from functools import lru_cache

import numpy as np, pandas as pd, matplotlib.pyplot as plt
import torch, soundfile as sf, librosa, transformers, datasets, huggingface_hub
from scipy import stats
from tqdm.auto import tqdm
from IPython.display import display
from datasets import load_dataset, Audio
from transformers import WhisperForConditionalGeneration, WhisperProcessor, WhisperTokenizer

transformers.logging.set_verbosity_error()          # keine Generierungs-/Tokenizer-Hinweise in den Ausgaben
datasets.utils.logging.set_verbosity_error()
huggingface_hub.utils.logging.set_verbosity_error()
if not torch.cuda.is_available():
    raise RuntimeError(
        f"CUDA ist nicht verfügbar (torch={torch.__version__}, CUDA-Build={torch.version.cuda}). "
        "Installiere die Abhängigkeiten aus requirements.txt (Abschnitt 0) und starte danach den Kernel neu."
    )
DEVICE = "cuda"
torch.cuda.set_device(0)
SEED = 0
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print("torch", torch.__version__, "| transformers", transformers.__version__, "| datasets", datasets.__version__, "|", DEVICE, "|",
      torch.cuda.get_device_name(0), "|", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1), "GiB VRAM")

QUICK = os.environ.get("WHISPER_XAI_QUICK", "0") == "1"   # Mini-Lauf zum Testen der Pipeline (eigener Ausgabeordner)
LOAD_CACHED = True                                         # gespeicherte Ergebnisse laden statt neu rechnen (nur bei identischer CFG)
CFG = dict(
    models        = ["openai/whisper-tiny", "openai/whisper-base"],
    n_utt         = 10 if QUICK else 100,   # Äußerungen für TF1–TF4
    n_faith       = 3  if QUICK else 25,    # Teilmenge für Deletion + Ablation (teuer)
    max_dur       = 15.0,                   # s
    ig_steps      = 32 if QUICK else 256,   # Stützstellen für Integrated Gradients (Vorprüfung: erst 256 konvergieren zuverlässig, Abschnitt 7c)
    ig_batch      = 4,                      # Stützstellen pro Backward-Batch (6-GB-GPU)
    ig_conv_steps = [8, 16, 32] if QUICK else [16, 32, 64, 128, 256],   # Konvergenzcheck
    n_conv        = 1 if QUICK else 3,      # Äußerungen für den Konvergenzcheck
    occ_win       = 0.10, occ_stride = 0.05,  # s, Zeit-Occlusion
    n_bands       = 8,                      # Frequenzband-Occlusion
    tol           = 0.02,                   # s, Toleranz Pointing Game
    ctx           = 0.30,                   # s, Kontextfenster vor/nach dem Wort
    del_fracs     = [0.25, 0.5, 1, 2, 4],   # entfernte Frames als Vielfaches der Wortlänge
    batch         = 4,                      # Forward-only-Perturbationen pro Batch
    amp           = False,                  # fp16-Autocast für Forward-only-Perturbationen (schneller, minimal ungenauer)
    ablation_wer  = True,                   # WER unter Kopf-Ablation (Generierung, langsamer)
    max_new_tokens= 200,
    n_random_sets = 5,                      # Zufallsmengen von Köpfen für die Gruppenablation
    loop_ratio    = 1.5,                    # Hypothese > 1,5 × Referenzlänge gilt als Schleife/Halluzination
    n_boot        = 200 if QUICK else 2000, # Bootstrap-Wiederholungen für Konfidenzintervalle
)
SR, HOP, ENC_HOP, N_MEL, N_FRAMES = 16000, 0.01, 0.02, 80, 3000   # Mel-Frame 10 ms, Encoder-Frame 20 ms
BINS = np.round(np.arange(-2.0, 3.01, 0.1), 2)   # relative Zeit (0 = Wortanfang, 1 = Wortende)
OUT = "results_quick" if QUICK else "results"
CACHE = os.path.join(OUT, "cache_res.pkl")
os.makedirs(OUT, exist_ok=True)
print("Modus:", "QUICK" if QUICK else "voll", "| Ausgabeordner:", OUT)

## 1 Daten: LibriSpeech `test-clean` mit Wortgrenzen

Wir nutzen `gilkeyio/librispeech-alignments` (Hugging Face): LibriSpeech-Audio (16 kHz) plus Wort- und Phonem-Alignments des **Montreal Forced Aligner** (McAuliffe et al., 2017; Alignments von Lugosch et al., 2019). Die MFA-Grenzen dienen als **Referenz („Ground Truth“)** für die Lokalisierung. Sie sind selbst modellbasiert und haben eine Unsicherheit von etwa einem Phonem; deshalb gibt es eine Toleranz `tol`.

**Textnormalisierung:** Referenz und Hypothese werden mit Whispers `EnglishTextNormalizer` verglichen (z. B. *Mr.* → *mister*, *colour* → *color*, Zahlwörter → Ziffern). Ohne diese Normalisierung würden reine Schreibvarianten als Fehler gezählt. Das erhöht die WER künstlich und schließt korrekt erkannte Wörter fälschlich aus der Analyse aus.

Es wird nur der Split `test_clean` gestreamt, nicht der ganze Datensatz. Audio wird selbst mit `soundfile` dekodiert, damit das unabhängig von der `datasets`-Version funktioniert.

In [ ]:
FUNCTION_WORDS = set("""a an the and or but if of to in on at by for with from as into onto upon
about than then so that this these those there here is are was were be been being am do does did
have has had i you he she it we they me him her us them my your his its our their mine not no nor
will would shall should can could may might must what which who whom whose when where why how
all any some such very just also too up out off over own same both each few more most other""".split())

_NORMALIZER = WhisperTokenizer.from_pretrained("openai/whisper-tiny")   # EnglishTextNormalizer, für alle Modelle identisch

def norm(w: str) -> str:
    """Grobe Normalisierung: klein, nur a-z, 0-9 und Apostroph."""
    return re.sub(r"[^a-z0-9']", "", w.lower())

@lru_cache(maxsize=None)
def match_key(w: str) -> str:
    """Vergleichsschlüssel eines Einzelworts nach Whispers Normalisierung (Mr. → mister, colour → color, can't → cannot)."""
    return norm(_NORMALIZER.normalize(w)) or norm(w)

def norm_words(text: str) -> list:
    """Wortliste nach Whispers EnglishTextNormalizer – Grundlage aller WER-Werte."""
    return _NORMALIZER.normalize(text).split()

def parse_words(w):
    """MFA-Wortliste -> [{word,start,end}] (verträgt list-of-dict und dict-of-lists)."""
    if isinstance(w, dict):
        w = [dict(zip(w.keys(), v)) for v in zip(*w.values())]
    out = [dict(word=norm(x["word"]), start=float(x["start"]), end=float(x["end"])) for x in w]
    return [x for x in out if x["word"] and x["end"] > x["start"]]

def load_utterances(n, max_dur, split="test_clean", seed=SEED):
    ds = load_dataset("gilkeyio/librispeech-alignments", split=split, streaming=True)
    ds = ds.cast_column("audio", Audio(decode=False)).shuffle(seed=seed, buffer_size=500)
    out = []
    for ex in ds:
        a = ex["audio"]
        wav, sr = sf.read(io.BytesIO(a["bytes"])) if a.get("bytes") else sf.read(a["path"])
        wav = wav.mean(1) if wav.ndim > 1 else wav
        if sr != SR:
            wav = librosa.resample(wav, orig_sr=sr, target_sr=SR)
        dur = len(wav) / SR
        words = parse_words(ex["words"])
        if dur > max_dur or len(words) < 3:
            continue
        out.append(dict(id=ex["id"], wav=wav.astype(np.float32), dur=dur, words=words))
        if len(out) >= n:
            break
    return out

def wer(ref, hyp):
    """Word Error Rate über Levenshtein-Distanz auf Wortebene (ohne Obergrenze, kann > 1 sein)."""
    d = np.arange(len(hyp) + 1)
    for i, r in enumerate(ref, 1):
        prev, d = d, d.copy(); d[0] = i
        for j, h in enumerate(hyp, 1):
            d[j] = min(prev[j] + 1, d[j - 1] + 1, prev[j - 1] + (r != h))
    return d[-1] / max(len(ref), 1)

def is_loop(ref_words, hyp_text, ratio):
    """Heuristik für Halluzination/Wiederholungsschleife: deutlich mehr Wörter als die Referenz
    oder stark komprimierbarer Text (Whispers eigenes Kriterium: Kompressionsrate > 2,4)."""
    b = hyp_text.encode("utf-8")
    cr = len(b) / max(len(zlib.compress(b)), 1)
    return bool(len(norm_words(hyp_text)) > ratio * max(len(ref_words), 1) or cr > 2.4)

def match_words(hyp, ref):
    """Index-Paare (hyp_i, ref_j) identischer Wörter in optimaler Ausrichtung (difflib)."""
    sm = difflib.SequenceMatcher(a=hyp, b=ref, autojunk=False)
    return [(b.a + k, b.b + k) for b in sm.get_matching_blocks() for k in range(b.size)]

assert match_key("Mr.") == match_key("mister") and match_key("colour") == match_key("color")
assert wer(norm_words("Mr. Smith's colour"), norm_words("mister smith's color")) == 0

In [ ]:
utts = load_utterances(CFG["n_utt"], CFG["max_dur"])
durs = np.array([u["dur"] for u in utts]); wd = np.array([w["end"] - w["start"] for u in utts for w in u["words"]])
print(f"{len(utts)} Äußerungen | Dauer {durs.mean():.1f}±{durs.std():.1f}s | {len(wd)} Wörter | Wortdauer Median {1000*np.median(wd):.0f} ms")

## 2 Modell-Wrapper

**Architektur (Radford et al., 2023):** Audio → 80-Kanal-Log-Mel-Spektrogramm (25 ms Fenster, 10 ms Hop), **immer auf 30 s aufgefüllt** (3000 Frames) → 2 Conv-Schichten (die zweite mit Stride 2) → Transformer-Encoder mit 1500 Frames à 20 ms → autoregressiver Decoder mit Cross-Attention auf den Encoder.

**Vorgehen pro Äußerung**
1. Whisper transkribiert greedy (Prompt `<|startoftranscript|><|en|><|transcribe|><|notimestamps|>`).
2. BPE-Tokens werden zu Wörtern gruppiert: Ein Token mit führendem Leerzeichen beginnt ein neues Wort, reine Satzzeichen-Tokens werden ignoriert.
3. Die erkannten Wörter werden nach Whisper-Normalisierung (Abschnitt 1) per `difflib` mit den Referenzwörtern ausgerichtet. Analysiert werden **nur korrekt erkannte Wörter**, denn nur für sie ist die MFA-Grenze die richtige Referenz.
4. Ein Forward-Pass mit **Teacher Forcing** auf die eigene Hypothese liefert $\log p(\text{Wort}) = \sum_{t\in\text{Wort}} \log p(y_t \mid y_{<t}, X)$. Diese Größe ist das Erklärungsziel $f(X)$ für alle Attributionsmethoden.

**Baseline $X_0$ („Stille“):** Das Padding-Spektrum der jeweiligen Äußerung. Der Feature-Extractor begrenzt Log-Mel-Werte nach unten auf (Maximum − 8). Aufgefüllte Nullen landen genau auf diesem Wert. Für das Modell ist $X_0$ also exakt das, was es als Stille kennt, und erzeugt keine Out-of-Distribution-Eingabe wie etwa ein Null-Tensor.

Es werden zwei Kopien des Modells geladen: `eager` für Attention-Gewichte und Hooks, `sdpa` für schnellere Gradienten und Perturbationen. Forward-only-Perturbationen laufen gebündelt (`batch`) und optional mit fp16-Autocast (`amp`, standardmäßig aus, damit kleine log-p-Differenzen nicht durch Rundung verfälscht werden).

In [ ]:
class WhisperXAI:
    def __init__(self, name):
        self.name = name
        self.proc = WhisperProcessor.from_pretrained(name)
        self.tok = self.proc.tokenizer
        load = lambda impl: WhisperForConditionalGeneration.from_pretrained(name, attn_implementation=impl).to(DEVICE).eval()
        self.model = load("eager")                       # Attention-Gewichte, Ablation-Hooks
        try:    self.model_g = load("sdpa")              # Gradienten / Perturbationen
        except Exception: self.model_g = self.model
        for m in {self.model, self.model_g}:
            for p in m.parameters(): p.requires_grad_(False)
        c = self.model.config
        self.L_enc, self.L_dec = c.encoder_layers, c.decoder_layers
        self.H_enc, self.H_dec = c.encoder_attention_heads, c.decoder_attention_heads
        self.prefix = self.tok.convert_tokens_to_ids(["<|startoftranscript|>", "<|en|>", "<|transcribe|>", "<|notimestamps|>"])
        self.eos = self.tok.eos_token_id                 # alle Spezialtokens haben IDs >= eos
        ah = getattr(self.model.generation_config, "alignment_heads", None)
        self.align = [tuple(int(v) for v in x) for x in ah] if ah else None
        print(f"{name}: Encoder {self.L_enc}x{self.H_enc}, Decoder {self.L_dec}x{self.H_dec}, alignment heads = {self.align}")

    def features(self, wav):
        return self.proc.feature_extractor(wav, sampling_rate=SR, return_tensors="pt").input_features.to(DEVICE)

    @torch.no_grad()
    def transcribe(self, feats, model=None):
        gen = (model or self.model_g).generate(feats, language="en", task="transcribe", max_new_tokens=CFG["max_new_tokens"])
        return [t for t in gen[0].tolist() if t < self.eos]   # nur Text-Tokens

    def token_words(self, text_ids):
        """[(wort_roh, [Indizes in text_ids])] – Satzzeichen-Tokens werden nicht als Wort-Tokens gezählt."""
        words = []
        for i, t in enumerate(text_ids):
            s = self.tok.decode([t])
            if s.startswith(" ") or not words:
                words.append(["", []])
            words[-1][0] += s
            if norm(s):
                words[-1][1].append(i)
        return [(w.strip(), idx) for w, idx in words if norm(w) and idx]

    def prepare(self, utt):
        feats = self.features(utt["wav"])
        text_ids = self.transcribe(feats)
        hyp = self.token_words(text_ids)
        ref = utt["words"]
        text, ref_text = self.tok.decode(text_ids), " ".join(r["word"] for r in ref)
        P = len(self.prefix)
        words = []
        for hi, ri in match_words([match_key(h[0]) for h in hyp], [match_key(r["word"]) for r in ref]):
            r = ref[ri]
            words.append(dict(word=r["word"], start=r["start"], end=r["end"], ref_idx=ri,
                              tok_pos=[P + i for i in hyp[hi][1]],   # Positionen in seq
                              func=r["word"] in FUNCTION_WORDS))
        return dict(id=utt["id"], dur=utt["dur"], ref_words=ref, ref_text=ref_text, feats=feats,
                    baseline=feats[..., -1:].expand_as(feats).clone(),   # Padding-Spektrum = Stille
                    seq=self.prefix + text_ids + [self.eos], words=words,
                    hyp_words=[match_key(h[0]) for h in hyp], text=text,
                    wer=wer(norm_words(ref_text), norm_words(text)),
                    T=min(int(np.ceil(utt["dur"] / HOP)), N_FRAMES),
                    n_a=min(int(np.ceil(utt["dur"] / ENC_HOP)), N_FRAMES // 2))

    def token_logprobs(self, model, feats, seq):
        """(B, len(seq)-1): log p(seq[j+1] | seq[:j+1], X) mit Teacher Forcing."""
        B = feats.shape[0]
        dec = torch.tensor(seq[:-1], device=DEVICE).unsqueeze(0).expand(B, -1)
        logits = model(input_features=feats, decoder_input_ids=dec).logits.float()
        tgt = torch.tensor(seq[1:], device=DEVICE).view(1, -1, 1).expand(B, -1, 1)
        return logits.gather(-1, tgt).squeeze(-1) - logits.logsumexp(-1)

    def word_logprobs(self, model, feats, seq, words):
        """(B, n_words): Summe der Token-Logprobs je Wort."""
        lp = self.token_logprobs(model, feats, seq)
        return torch.stack([lp[:, [p - 1 for p in w["tok_pos"]]].sum(-1) for w in words], -1)

    @torch.no_grad()
    def lp_eval(self, feats, prep):
        """Forward-only log p je Wort, optional mit fp16-Autocast (CFG['amp']). Referenz und Perturbationen nutzen dieselbe Funktion."""
        with (torch.autocast("cuda", dtype=torch.float16) if CFG["amp"] else nullcontext()):
            return self.word_logprobs(self.model_g, feats, prep["seq"], prep["words"]).float()

## 3 Input-Attribution

Alle Methoden erklären $f(X)=\log p(\text{Wort}\mid X)$ bezüglich des Log-Mel-Spektrogramms $X\in\mathbb{R}^{80\times3000}$.

* **Integrated Gradients** (Sundararajan et al., 2017): $\mathrm{IG}_i = (x_i - x_{0,i}) \int_0^1 \frac{\partial f(x_0+\alpha(x-x_0))}{\partial x_i}\,d\alpha$. Das Integral wird mit der Mittelpunktsregel über `ig_steps` = 256 Stützstellen angenähert. Nach dem Axiom der *Completeness* gilt $\sum_i \mathrm{IG}_i = f(x)-f(x_0)$. Daran prüfen wir pro Wort, ob die Näherung konvergiert ist; Abschnitt 7c zeigt zusätzlich den Fehler in Abhängigkeit von der Schrittzahl (16 bis 256). Neben IG wird der mittlere Gradient entlang des Pfades $\bar g = \int_0^1 \nabla f\,d\alpha$ gespeichert. Er enthält den Faktor $(x-x_0)$ nicht und dient in Abschnitt 9 als energiefreie Vergleichsgröße.
* **Gradient × Δ** (Shrikumar et al., 2016, mit Baseline): $\nabla f(x)\odot(x-x_0)$. Entspricht IG mit einer einzigen Stützstelle bei $\alpha=1$, ist also lokal und billig, verletzt aber bei Sättigung die Completeness.
* **Zeit-Occlusion** (Zeiler & Fergus, 2014): Ein 100-ms-Fenster wird auf $x_0$ gesetzt und der Abfall von $\log p$ gemessen. Modellagnostisch und kausal. Ein Forward-Pass liefert alle Wörter gleichzeitig. Überlappende Fenster werden pro Frame gemittelt.
* **Frequenzband-Occlusion:** 8 Mel-Bandgruppen werden *nur innerhalb des Wortsegments* (±50 ms) auf $x_0$ gesetzt. Zusätzlich wird die entfernte Energie $\sum (x-x_0)$ je Band gespeichert, damit der Effekt auch pro Energieeinheit angegeben werden kann.

Für die Zeitlokalisierung wird $|{\rm Attribution}|$ über die Mel-Achse summiert. Bei Occlusion wird nur der positive Teil verwendet, also Fenster, deren Entfernen das Wort unwahrscheinlicher macht.

In [ ]:
def integrated_gradients(xai, prep, steps, batch):
    """IG (Mittelpunktsregel). Rückgabe: IG (W, 80, 3000), relativer Completeness-Fehler je Wort, mittlerer Gradient ḡ (W, 80, 3000)."""
    x, x0, seq, words = prep["feats"], prep["baseline"], prep["seq"], prep["words"]
    W = len(words)
    total = torch.zeros(W, *x.shape[1:], device=DEVICE)
    alphas = (torch.arange(steps, device=DEVICE) + 0.5) / steps
    for a in alphas.split(batch):
        xi = (x0 + a.view(-1, 1, 1) * (x - x0)).requires_grad_(True)
        lp = xai.word_logprobs(xai.model_g, xi, seq, words)                 # (b, W)
        for w in range(W):
            g, = torch.autograd.grad(lp[:, w].sum(), xi, retain_graph=w < W - 1)
            total[w] += g.sum(0)
    gbar = total / steps                                                   # mittlerer Gradient entlang des Pfades
    ig = (x - x0)[0] * gbar
    with torch.no_grad():                                                  # Completeness-Check
        lp_end = xai.word_logprobs(xai.model_g, torch.cat([x, x0]), seq, words)
    delta = (lp_end[0] - lp_end[1])
    rel_err = ((ig.sum((1, 2)) - delta).abs() / delta.abs().clamp_min(1e-6)).cpu().numpy()
    return ig, rel_err, gbar

def ig_convergence(xai, prep, steps_list, batch):
    """Completeness-Fehler je Wort für verschiedene Schrittzahlen (Abschnitt 7c)."""
    rows = []
    for s in steps_list:
        _, err, _ = integrated_gradients(xai, prep, s, batch)
        rows += [dict(model=xai.name, utt=prep["id"], steps=s, word=w["word"], rel_err=e) for w, e in zip(prep["words"], err)]
        torch.cuda.empty_cache()
    return rows

def grad_times_delta(xai, prep):
    x = prep["feats"].clone().requires_grad_(True)
    lp = xai.word_logprobs(xai.model_g, x, prep["seq"], prep["words"])[0]
    out = []
    for w in range(len(prep["words"])):
        g, = torch.autograd.grad(lp[w], x, retain_graph=w < len(prep["words"]) - 1)
        out.append((g * (prep["feats"] - prep["baseline"]))[0])   # ohne Autograd-Graph
    return torch.stack(out)                                                # (W, 80, 3000)

@torch.no_grad()
def run_batched(xai, prep, edits, batch):
    """edits: Liste von Funktionen xb_j -> None (in-place). Gibt (n_edits, W) log p zurück."""
    x, out = prep["feats"], []
    for i in range(0, len(edits), batch):
        chunk = edits[i:i + batch]
        xb = x.repeat(len(chunk), 1, 1)
        for j, e in enumerate(chunk):
            e(xb[j])
        out.append(xai.lp_eval(xb, prep))
    return torch.cat(out).cpu().numpy()

def occlusion_time(xai, prep, win, stride, batch):
    x0, T = prep["baseline"][0], prep["T"]
    wf, sf_ = int(round(win / HOP)), int(round(stride / HOP))
    starts = list(range(0, max(T - wf, 0) + 1, sf_))
    def edit(s):
        def e(xb): xb[:, s:s + wf] = x0[:, s:s + wf]
        return e
    ref = xai.lp_eval(prep["feats"], prep)[0].cpu().numpy()
    drops = ref[None] - run_batched(xai, prep, [edit(s) for s in starts], batch)   # (n_win, W)
    prof, cnt = np.zeros((len(prep["words"]), T)), np.zeros(T)
    for j, s in enumerate(starts):
        prof[:, s:s + wf] += drops[j][:, None]; cnt[s:s + wf] += 1
    return prof / np.maximum(cnt, 1)                                        # (W, T)

def occlusion_bands(xai, prep, n_bands, batch, pad=0.05):
    """Abfall von log p je (Wort, Band) und die dabei entfernte Energie Σ(x − x0)."""
    x, x0, T, words = prep["feats"][0], prep["baseline"][0], prep["T"], prep["words"]
    edges = np.linspace(0, N_MEL, n_bands + 1).astype(int)
    edits, idx = [], []
    energy = np.zeros((len(words), n_bands))
    for wi, w in enumerate(words):
        a = int(max(0.0, w["start"] - pad) / HOP); b = min(T, int(np.ceil((w["end"] + pad) / HOP)))
        for bi in range(n_bands):
            lo, hi = edges[bi], edges[bi + 1]
            def e(xb, a=a, b=b, lo=lo, hi=hi): xb[lo:hi, a:b] = x0[lo:hi, a:b]
            edits.append(e); idx.append((wi, bi))
            energy[wi, bi] = (x[lo:hi, a:b] - x0[lo:hi, a:b]).sum().item()
    ref = xai.lp_eval(prep["feats"], prep)[0].cpu().numpy()
    lp = run_batched(xai, prep, edits, batch)
    res = np.zeros((len(words), n_bands))
    for j, (wi, bi) in enumerate(idx):
        res[wi, bi] = ref[wi] - lp[j, wi]
    return res, edges, energy

## 4 Attention: Cross-Attention des Decoders und Self-Attention des Encoders

* **Cross-Attention:** Für ein Wort werden die Zeilen der Decoder-Queries gemittelt, die seine Tokens vorhersagen (Position $p-1$ für Token $p$). Ergebnis ist eine Verteilung über 1500 Encoder-Frames pro Schicht und Kopf. Whisper selbst nutzt ausgewählte Köpfe (`alignment_heads`) mit Dynamic Time Warping für Wort-Zeitstempel. Wir prüfen, ob gerade diese Köpfe gut lokalisieren.
* **Encoder-Self-Attention:** Zeilen = Queries im Audiobereich. Pro Kopf berechnen wir
  - `local`: Masse im Bereich |i−j| ≤ 2 Frames (±40 ms),
  - `prev`/`next`: Masse auf dem direkt vorherigen bzw. folgenden Frame,
  - `mean_dist`: mittlere Aufmerksamkeitsdistanz in s,
  - `entropy`: normiert auf [0, 1],
  - `pad_mass`: Masse auf dem 30-s-Padding, ein möglicher *Attention Sink*,
  - `within_lift`: Anteil der Masse im *selben Wort* geteilt durch den Zufallswert. Werte > 1 bedeuten eine Bindung an Wortgrenzen.
  - `sil_lift`: Masse auf Pausen-Frames geteilt durch deren Anteil.

  Die Metriken folgen Clark et al. (2019) und Voita et al. (2019).

**Wichtig:** Attention-Gewichte zeigen, wie Information verteilt wird. Sie sind *keine* Attribution, siehe Jain & Wallace (2019) und Wiegreffe & Pinter (2019). Deshalb prüfen wir in Abschnitt 12 kausal per Ablation.

In [ ]:
@torch.no_grad()
def get_attentions(xai, prep):
    dec = torch.tensor(prep["seq"][:-1], device=DEVICE)[None]
    out = xai.model(input_features=prep["feats"], decoder_input_ids=dec, output_attentions=True)
    cross = torch.stack(out.cross_attentions)[:, 0].float()      # (L_dec, H, len-1, 1500)
    enc = torch.stack(out.encoder_attentions)[:, 0].float()      # (L_enc, H, 1500, 1500)
    return cross, enc

def frame_labels(words, n, hop):
    """Wortindex je Frame (-1 = Pause/Stille)."""
    lab = np.full(n, -1); t = (np.arange(n) + 0.5) * hop
    for k, w in enumerate(words):
        lab[(t >= w["start"]) & (t < w["end"])] = k
    return lab

@torch.no_grad()
def encoder_head_stats(enc, n_a, words):
    A = enc[:, :, :n_a, :]                                         # Queries im Audio
    pad_mass = A[..., n_a:].sum(-1).mean(-1)
    Aa = A[..., :n_a]; Aa = Aa / Aa.sum(-1, keepdim=True).clamp_min(1e-12)
    i = torch.arange(n_a, device=Aa.device)
    D = (i[:, None] - i[None, :]).abs().float()
    lab = torch.tensor(frame_labels(words, n_a, ENC_HOP), device=Aa.device)
    same = ((lab[:, None] == lab[None, :]) & (lab[:, None] >= 0)).float()
    q_in = lab >= 0; sil = (lab < 0).float()
    within = (Aa * same).sum(-1)[..., q_in].mean(-1)
    within_chance = (same.sum(-1) / n_a)[q_in].mean()
    s = dict(
        local=(Aa * (D <= 2)).sum(-1).mean(-1),
        prev=Aa.diagonal(offset=-1, dim1=-2, dim2=-1).mean(-1),     # Query i -> Key i-1
        next=Aa.diagonal(offset=1, dim1=-2, dim2=-1).mean(-1),      # Query i -> Key i+1
        mean_dist=(Aa * D).sum(-1).mean(-1) * ENC_HOP,
        entropy=-(Aa * Aa.clamp_min(1e-12).log()).sum(-1).mean(-1) / math.log(n_a),
        pad_mass=pad_mass,
        within_lift=within / within_chance.clamp_min(1e-9),
        sil_lift=(Aa * sil).sum(-1).mean(-1) / sil.mean().clamp_min(1e-9),
    )
    return {k: v.cpu().numpy() for k, v in s.items()}

## 5 Lokalisierungsmetriken

Für ein nichtnegatives, auf Summe 1 normiertes Zeitprofil $p(t)$ über den Audiobereich und ein Wort $[s, e)$:

| Metrik | Definition | Zufallsniveau |
|---|---|---|
| `mass_in` | $\sum_{t\in[s,e)} p(t)$ | `chance` = Wortdauer / Äußerungsdauer |
| `lift` | `mass_in / chance` | 1 |
| `hit` (Pointing Game) | $\arg\max_t p(t) \in [s-\text{tol}, e+\text{tol})$ | `hit_chance` |
| `before` / `after` | Masse in den 300 ms vor bzw. nach dem Wort | – |
| `peak_off` | Peak-Zeit minus Wortmitte (s) | 0 |

Zusätzlich gibt es ein **ereigniskorreliertes Profil**: Die Zeit wird relativ zum Wort normiert ($u=(t-s)/(e-s)$, also 0 = Anfang, 1 = Ende) und die Relevanz pro Frame relativ zur Gleichverteilung über alle Wörter gemittelt. Damit sieht man, ob die Evidenz am Wortanfang, in der Mitte oder im Nachlauf (Koartikulation) liegt.

**Statistik.** Wörter derselben Äußerung sind nicht unabhängig (gleicher Sprecher, gleiche Aufnahme, gemeinsamer Decoder-Kontext). Deshalb gilt durchgehend:
* Tests laufen auf **Äußerungsebene**: Werte werden zuerst je Äußerung gemittelt, dann wird über die Äußerungen getestet (n = 100 bzw. 25).
* Konfidenzintervalle (95 %) stammen aus einem **Cluster-Bootstrap**: Äußerungen werden mit Zurücklegen gezogen, ihre Wörter bleiben zusammen.
* Bei mehreren gleichartigen Tests wird nach **Holm** korrigiert.

In [ ]:
def loc_metrics(profile, hop, start, end, dur, tol, ctx):
    p = np.clip(np.asarray(profile, dtype=np.float64), 0, None)
    if p.sum() <= 0:
        return None
    p = p / p.sum()
    t = (np.arange(len(p)) + 0.5) * hop
    inw = (t >= start) & (t < end)
    chance = max(inw.mean(), 1e-9)
    peak = t[p.argmax()]
    return dict(mass_in=p[inw].sum(), chance=chance, lift=p[inw].sum() / chance,
                hit=float(start - tol <= peak < end + tol),
                hit_chance=min(1.0, (end - start + 2 * tol) / dur),
                before=p[(t >= start - ctx) & (t < start)].sum(),
                after=p[(t >= end) & (t < end + ctx)].sum(),
                peak_off=peak - (start + end) / 2)

def aligned_profile(profile, hop, start, end, bins=BINS):
    """Mittlere Relevanz je relativer Zeit-Bin, normiert auf Gleichverteilung (1 = Zufall)."""
    p = np.clip(np.asarray(profile, dtype=np.float64), 0, None)
    if p.sum() <= 0:
        return np.full(len(bins) - 1, np.nan)
    p = p / p.sum() * len(p)
    u = ((np.arange(len(p)) + 0.5) * hop - start) / (end - start)
    b = np.digitize(u, bins) - 1
    ok = (b >= 0) & (b < len(bins) - 1)
    s = np.bincount(b[ok], weights=p[ok], minlength=len(bins) - 1)
    c = np.bincount(b[ok], minlength=len(bins) - 1)
    return np.where(c > 0, s / np.maximum(c, 1), np.nan)

# ---- Statistik-Helfer
def boot_ci(df, col, by="utt", n=None, seed=SEED):
    """95-%-Cluster-Bootstrap-KI des Mittelwerts von `col`: Äußerungen werden gezogen, ihre Wörter bleiben zusammen."""
    g = df.groupby(by)[col].agg(["sum", "count"])
    s, c = g["sum"].to_numpy(float), g["count"].to_numpy(float)
    idx = np.random.default_rng(seed).integers(0, len(s), size=(n or CFG["n_boot"], len(s)))
    return np.quantile(s[idx].sum(1) / c[idx].sum(1), [0.025, 0.975])

def ols_boot(df, y, xcols, by="utt", n=None, seed=SEED):
    """OLS y ~ 1 + xcols; 95-%-KI der Koeffizienten per Cluster-Bootstrap über Äußerungen."""
    X = np.column_stack([np.ones(len(df))] + [df[c].to_numpy(float) for c in xcols]); Y = df[y].to_numpy(float)
    beta = np.linalg.lstsq(X, Y, rcond=None)[0]
    codes = pd.factorize(df[by])[0]
    groups = [np.flatnonzero(codes == k) for k in range(codes.max() + 1)]
    rng, bs = np.random.default_rng(seed), []
    for _ in range(n or CFG["n_boot"]):
        ix = np.concatenate([groups[k] for k in rng.integers(0, len(groups), len(groups))])
        bs.append(np.linalg.lstsq(X[ix], Y[ix], rcond=None)[0])
    lo, hi = np.quantile(np.array(bs), [0.025, 0.975], axis=0)
    return pd.DataFrame(dict(coef=beta, ci_lo=lo, ci_hi=hi), index=["Achsenabschnitt"] + list(xcols))

def holm(p):
    """Holm-Bonferroni-korrigierte p-Werte."""
    p = np.asarray(p, float); adj, run = np.empty(len(p)), 0.0
    for r, i in enumerate(np.argsort(p)):
        run = max(run, (len(p) - r) * p[i]); adj[i] = min(1.0, run)
    return adj

# Plausibilitätstests: synthetisches Profil (Gauß-Peak in der Wortmitte) und Holm-Korrektur
_t = (np.arange(1000) + 0.5) * HOP
_m = loc_metrics(np.exp(-((_t - 5.0) / 0.05) ** 2), HOP, 4.8, 5.2, 10.0, 0.02, 0.3)
assert _m["hit"] == 1 and _m["mass_in"] > 0.99 and abs(_m["peak_off"]) < 0.01
assert np.allclose(holm([0.01, 0.04, 0.03]), [0.03, 0.06, 0.06])
print({k: round(float(v), 3) for k, v in _m.items()})

## 6 Deletion-Test und Kopf-Ablation (Faithfulness)

**Deletion** (Samek et al., 2017): Pro Wort werden die nach der jeweiligen Methode relevantesten Mel-Frames auf $x_0$ gesetzt, und zwar $k \in \{0.25, 0.5, 1, 2, 4\}\times$ Wortlänge viele. Gemessen wird $\Delta\log p(\text{Wort})$. Eine treue Erklärung lässt $\log p$ schneller fallen. Vergleiche:
* `Zufall` als untere Schranke,
* `Oracle`: Frames nach Abstand zur Wortmitte, also „die Wahrheit laut MFA“.

Gleich bewertete Frames (z. B. die vielen Nullen der Occlusion nach dem Abschneiden negativer Werte) werden **zufällig** statt nach Position sortiert. Sonst würden bei großen Anteilen systematisch die ersten Frames der Äußerung entfernt.

**Kopf-Ablation** (Michel et al., 2019): Die Ausgabe eines Kopfes wird vor der Output-Projektion auf 0 gesetzt, per `forward_pre_hook` auf `out_proj`. Gemessen werden $\Delta$ (mittlerer Token-$\log p$) bei Teacher Forcing und die WER bei freier Generierung. Die WER wird nach Whisper-Normalisierung berechnet, und zwar sowohl ungekappt als auch **je Äußerung auf 1 gekappt**. Die Referenz-WER stammt aus derselben `eager`-Modellkopie wie die ablatierte Generierung. Alle generierten Texte werden gespeichert; Schleifen/Halluzinationen werden markiert (Hypothese > 1,5 × Referenzlänge oder Kompressionsrate > 2,4). Zusätzlich werden alle `alignment_heads` gemeinsam abgeschaltet und mit gleich großen Zufallsmengen von Köpfen verglichen.

In [ ]:
def deletion_test(xai, prep, profiles, fracs, batch, rng):
    """profiles: {Methode: (W, T)} -> Zeilen mit Δ log p je (Methode, Wort, Anteil)."""
    x0, T, words = prep["baseline"][0], prep["T"], prep["words"]
    t = (np.arange(T) + 0.5) * HOP
    profiles = dict(profiles)
    profiles["Oracle"] = np.stack([-np.abs(t - (w["start"] + w["end"]) / 2) for w in words])
    profiles["Zufall"] = rng.random((len(words), T))
    tie = rng.random((len(words), T))                                      # zufällige Auflösung von Gleichständen
    ref = xai.lp_eval(prep["feats"], prep)[0].cpu().numpy()
    edits, meta = [], []
    for m, P in profiles.items():
        for wi, w in enumerate(words):
            order = np.lexsort((tie[wi], -P[wi]))                         # absteigend nach Relevanz, Gleichstand zufällig
            wf = max(1, int(round((w["end"] - w["start"]) / HOP)))
            for f in fracs:
                k = min(T, max(1, int(round(f * wf))))
                idx = torch.as_tensor(order[:k], dtype=torch.long, device=DEVICE)
                def e(xb, idx=idx): xb[:, idx] = x0[:, idx]
                edits.append(e); meta.append((m, wi, f))
    lp = run_batched(xai, prep, edits, batch)
    return [dict(model=xai.name, utt=prep["id"], method=m, wi=wi, word=words[wi]["word"], frac=f,
                 delta=lp[j, wi] - ref[wi]) for j, (m, wi, f) in enumerate(meta)]

@contextmanager
def ablate_heads(xai, heads, kind):
    """Setzt die Ausgabe der Köpfe [(layer, head)] auf 0. kind: 'enc_self' | 'dec_cross'."""
    hooks = []
    for l, h in heads:
        mod = (xai.model.model.encoder.layers[l].self_attn if kind == "enc_self"
               else xai.model.model.decoder.layers[l].encoder_attn)
        d = mod.out_proj.in_features // (xai.H_enc if kind == "enc_self" else xai.H_dec)
        def pre(m, args, h=h, d=d):
            z = args[0].clone(); z[..., h * d:(h + 1) * d] = 0
            return (z,) + tuple(args[1:])
        hooks.append(mod.out_proj.register_forward_pre_hook(pre))
    try:
        yield
    finally:
        for hk in hooks: hk.remove()

@torch.no_grad()
def mean_text_logprob(xai, prep):
    P, n = len(xai.prefix), len(prep["seq"])
    return xai.token_logprobs(xai.model, prep["feats"], prep["seq"])[0, P - 1:n - 2].mean().item()

def head_ablation(xai, preps, kind, head_sets, do_wer, loop_ratio):
    refs = [norm_words(p["ref_text"]) for p in preps]
    gen = lambda: [xai.tok.decode(xai.transcribe(p["feats"], model=xai.model)) for p in preps]   # immer eager-Modell
    werv = lambda texts: np.array([wer(r, norm_words(t)) for r, t in zip(refs, texts)])
    base_lp = [mean_text_logprob(xai, p) for p in preps]
    if do_wer:
        base_w = werv(gen())
    rows = []
    for label, heads in tqdm(head_sets, desc=f"Ablation {kind}", leave=False):
        with ablate_heads(xai, heads, kind):
            row = dict(model=xai.name, kind=kind, label=label, heads=[tuple(map(int, x)) for x in heads],
                       d_logprob=np.mean([mean_text_logprob(xai, p) - b for p, b in zip(preps, base_lp)]))
            if do_wer:
                texts = gen()
                w = werv(texts)
                row.update(wer=w.mean(), d_wer=w.mean() - base_w.mean(),
                           wer_capped=np.minimum(w, 1).mean(), d_wer_capped=np.minimum(w, 1).mean() - np.minimum(base_w, 1).mean(),
                           n_loop=sum(is_loop(r, t, loop_ratio) for r, t in zip(refs, texts)),
                           wers=w, texts=texts, utt_ids=[p["id"] for p in preps])
        rows.append(row)
    return rows

## 7 Hauptschleife

Pro Modell und Äußerung: Vorbereitung → IG, Grad×Δ, Zeit- und Band-Occlusion → Attention → Metriken pro Wort und Methode. Für die ersten `n_faith` Äußerungen zusätzlich der Deletion-Test und am Ende die Kopf-Ablation, für die ersten `n_conv` Äußerungen der IG-Konvergenzcheck.

Die Ergebnisse werden mit der `CFG` in `results/cache_res.pkl` gespeichert und bei identischer `CFG` wieder geladen (`LOAD_CACHED`, Abschnitt 0). Die Modelle müssen dann nicht neu gerechnet werden.

In [ ]:
def run_analysis(name, utts, cfg=CFG):
    xai = WhisperXAI(name)
    rng = np.random.default_rng(SEED)
    R = defaultdict(list)
    Ld, Hd = xai.L_dec, xai.H_dec
    cross_acc = dict(hit=np.zeros((Ld, Hd)), mass=np.zeros((Ld, Hd)), lift=np.zeros((Ld, Hd)), pad=np.zeros((Ld, Hd)))
    n_cross, faith_preps, example, edges = 0, [], None, None

    for ui, utt in enumerate(tqdm(utts, desc=name)):
        prep = xai.prepare(utt)
        R["utt"].append(dict(model=name, utt=prep["id"], dur=prep["dur"], n_ref=len(prep["ref_words"]),
                             n_matched=len(prep["words"]), ref_text=prep["ref_text"], text=prep["text"], wer=prep["wer"]))
        if not prep["words"]:
            continue
        T, n_a, words = prep["T"], prep["n_a"], prep["words"]

        # ---- Attribution
        ig, ig_err, gbar = integrated_gradients(xai, prep, cfg["ig_steps"], cfg["ig_batch"])
        if ui < cfg["n_conv"]:
            R["ig_conv"] += ig_convergence(xai, prep, cfg["ig_conv_steps"], cfg["ig_batch"])
        gxd = grad_times_delta(xai, prep)
        occ = occlusion_time(xai, prep, cfg["occ_win"], cfg["occ_stride"], cfg["batch"])
        bands, edges, band_en = occlusion_bands(xai, prep, cfg["n_bands"], cfg["batch"])
        ig_abs = ig.abs()
        ig_t = ig_abs.sum(1)[:, :T].cpu().numpy()
        gxd_t = gxd.abs().sum(1)[:, :T].cpu().numpy()
        delta_x = (prep["feats"] - prep["baseline"])[0]

        # ---- Attention
        cross, enc = get_attentions(xai, prep)
        for k, v in encoder_head_stats(enc, n_a, prep["ref_words"]).items():
            R["enc_" + k].append(v)
        if example is None:
            example = dict(prep={k: (v.cpu() if torch.is_tensor(v) else v) for k, v in prep.items() if k != "baseline"},
                           ig=ig.cpu().numpy(), enc=enc[:, :, :n_a, :n_a].half().cpu().numpy(),
                           cross=cross[:, :, :, :n_a].cpu().numpy(),
                           tokens=[xai.tok.decode([t]) for t in prep["seq"][1:]])
        del enc

        prof_mel = defaultdict(list)    # für Deletion: Profile in Mel-Frames
        for wi, w in enumerate(words):
            rows = [p - 1 for p in w["tok_pos"]]
            c = cross[:, :, rows, :].mean(2)                               # (Ld, Hd, 1500)
            ca = c[..., :n_a]
            cross_acc["pad"] += (1 - ca.sum(-1)).cpu().numpy()
            pn = (ca / ca.sum(-1, keepdim=True).clamp_min(1e-12)).cpu().numpy()
            te = (np.arange(n_a) + 0.5) * ENC_HOP
            inw = (te >= w["start"]) & (te < w["end"])
            peak = te[pn.argmax(-1)]
            cross_acc["hit"] += (peak >= w["start"] - cfg["tol"]) & (peak < w["end"] + cfg["tol"])
            cross_acc["mass"] += pn[..., inw].sum(-1)
            cross_acc["lift"] += pn[..., inw].sum(-1) / max(inw.mean(), 1e-9)
            n_cross += 1

            profiles = {"IG": (ig_t[wi], HOP), "Grad×Δ": (gxd_t[wi], HOP),
                        "Occlusion": (np.clip(occ[wi], 0, None), HOP),
                        "CrossAttn (alle Köpfe)": (pn.mean((0, 1)), ENC_HOP)}
            if xai.align:
                profiles["CrossAttn (Alignment-Köpfe)"] = (np.mean([pn[l, h] for l, h in xai.align], 0), ENC_HOP)

            base = dict(model=name, utt=prep["id"], word=w["word"], func=w["func"], w_dur=w["end"] - w["start"],
                        n_tok=len(w["tok_pos"]), pos=w["ref_idx"], ig_err=ig_err[wi])
            for m, (p, hop) in profiles.items():
                met = loc_metrics(p, hop, w["start"], w["end"], prep["dur"], cfg["tol"], cfg["ctx"])
                if met:
                    R["words"].append(dict(base, method=m, **met))
                R["erp"].append(dict(model=name, method=m, func=w["func"], prof=aligned_profile(p, hop, w["start"], w["end"])))
                prof_mel[m].append(p if hop == HOP else np.repeat(p, 2)[:T])

            # ---- spektral: |IG|, |ḡ| (ohne Energiefaktor) und Energie (x - x0) je Mel-Bin im Wortsegment
            a = int(w["start"] / HOP); b = max(a + 1, int(np.ceil(w["end"] / HOP)))
            nrm = lambda v: (v / v.sum().clamp_min(1e-12)).cpu().numpy()
            R["spec"].append(dict(model=name, utt=prep["id"], func=w["func"], ig=nrm(ig_abs[wi, :, a:b].sum(-1)),
                                  grad=nrm(gbar[wi, :, a:b].abs().mean(-1)), energy=nrm(delta_x[:, a:b].mean(-1))))
            R["bands"].append(dict(model=name, utt=prep["id"], func=w["func"], word=w["word"],
                                   **{f"b{k}": bands[wi, k] for k in range(len(edges) - 1)},
                                   **{f"e{k}": band_en[wi, k] for k in range(len(edges) - 1)}))

        if ui < cfg["n_faith"]:
            R["deletion"] += deletion_test(xai, prep, {m: np.stack(v) for m, v in prof_mel.items()},
                                           cfg["del_fracs"], cfg["batch"], rng)
            faith_preps.append(prep)
        del cross, ig, ig_abs, gxd, gbar
        torch.cuda.empty_cache()

    # ---- Kopf-Ablation
    enc_sets = [(f"E{l}.{h}", [(l, h)]) for l in range(xai.L_enc) for h in range(xai.H_enc)]
    dec_sets = [(f"C{l}.{h}", [(l, h)]) for l in range(Ld) for h in range(Hd)]
    if xai.align:
        allh = [(l, h) for l in range(Ld) for h in range(Hd)]
        dec_sets.append(("alignment heads", xai.align))
        for r in range(cfg["n_random_sets"]):
            pick = sorted(rng.choice(len(allh), size=len(xai.align), replace=False))
            dec_sets.append((f"Zufall {r}", [allh[i] for i in pick]))
    R["ablation"] = (head_ablation(xai, faith_preps, "enc_self", enc_sets, cfg["ablation_wer"], cfg["loop_ratio"]) +
                     head_ablation(xai, faith_preps, "dec_cross", dec_sets, cfg["ablation_wer"], cfg["loop_ratio"]))

    out = dict(name=name, align=xai.align, L_enc=xai.L_enc, H_enc=xai.H_enc, L_dec=Ld, H_dec=Hd, example=example,
               band_edges=edges, cross={k: v / max(n_cross, 1) for k, v in cross_acc.items()},
               enc={k[4:]: np.mean(v, 0) for k, v in R.items() if k.startswith("enc_")},
               enc_utt={k[4:]: np.stack(v) for k, v in R.items() if k.startswith("enc_")})   # (n_utt, L, H) für H3
    for k in ("utt", "words", "erp", "spec", "bands", "deletion", "ablation", "ig_conv"):
        out[k] = pd.DataFrame(R[k])
    del xai, faith_preps
    torch.cuda.empty_cache()
    return out

In [ ]:
cfg_key = lambda c: json.dumps(c, sort_keys=True, default=str)
cached = None
if LOAD_CACHED and os.path.exists(CACHE):
    with open(CACHE, "rb") as f:
        cached = pickle.load(f)
    if cfg_key(cached["cfg"]) != cfg_key(CFG):
        print("Cache passt nicht zur aktuellen CFG – es wird neu gerechnet."); cached = None
if cached:
    RES = cached["res"]
    print(f"Ergebnisse aus {CACHE} geladen (erzeugt {cached['created']}, Laufzeit {cached['runtime_min']:.0f} min).")
else:
    t0 = pd.Timestamp.now()
    RES = {name: run_analysis(name, utts) for name in CFG["models"]}
    runtime = (pd.Timestamp.now() - t0).total_seconds() / 60
    with open(CACHE, "wb") as f:
        pickle.dump(dict(cfg=CFG, created=t0.isoformat(timespec="seconds"), runtime_min=runtime, res=RES), f)
    print(f"Laufzeit {runtime:.0f} min, Ergebnisse gespeichert in {CACHE}")

for name, r in RES.items():
    r["words"].to_csv(f"{OUT}/{name.split('/')[-1]}_word_metrics.csv", index=False)
    e = r["words"].query("method == 'IG'").drop_duplicates(["utt", "pos"]).ig_err
    print(f"{name}: WER {r['utt'].wer.mean():.3f} | {r['utt'].n_matched.sum()} korrekt erkannte Wörter | "
          f"IG-Completeness ({CFG['ig_steps']} Schritte): Median {e.median():.3f}, Anteil < 5 %: {(e < 0.05).mean():.2f}")
MAIN = CFG["models"][0]
MODELS = list(RES)

## 7b Beispiel-Äußerung

Gezeigt werden Spektrogramm mit MFA-Grenzen, IG-Karte für das längste Inhaltswort, Zeitprofile mehrerer Methoden für dieses Wort und die Cross-Attention der Alignment-Köpfe über alle Tokens.

In [ ]:
ex = RES[MAIN]["example"]; P = ex["prep"]; T, n_a = P["T"], P["n_a"]
wi = int(np.argmax([(w["end"] - w["start"]) * (not w["func"]) for w in P["words"]])); W = P["words"][wi]
print("Hypothese:", P["text"]); print("Fokuswort:", W["word"], f"[{W['start']:.2f}, {W['end']:.2f}] s")

def mark_words(ax, words, ymax=None, label=True):
    for w in words:
        ax.axvspan(w["start"], w["end"], color="tab:orange", alpha=0.08)
        ax.axvline(w["start"], color="k", lw=0.4, alpha=0.4)
        if label and ymax is not None:
            ax.text((w["start"] + w["end"]) / 2, ymax, w["word"], ha="center", va="bottom", fontsize=7, rotation=45)

fig, ax = plt.subplots(4, 1, figsize=(15, 13), sharex=True, gridspec_kw=dict(height_ratios=[1, 1, 1, 1.4]))
ax[0].imshow(P["feats"][0, :, :T].cpu(), origin="lower", aspect="auto", extent=[0, T * HOP, 0, N_MEL], cmap="magma")
mark_words(ax[0], P["ref_words"], N_MEL); ax[0].set_ylabel("Mel-Bin"); ax[0].set_title("Log-Mel-Spektrogramm + MFA-Wortgrenzen", pad=28)
m = np.abs(ex["ig"][wi][:, :T]); ax[1].imshow(m, origin="lower", aspect="auto", extent=[0, T * HOP, 0, N_MEL], cmap="viridis", vmax=np.quantile(m, 0.999))
ax[1].axvspan(W["start"], W["end"], fill=False, ec="r", lw=1.5); ax[1].set_ylabel("Mel-Bin"); ax[1].set_title(f"|IG| für »{W['word']}«")
t = (np.arange(T) + 0.5) * HOP
ig_prof = np.abs(ex["ig"][wi]).sum(0)[:T]; ax[2].plot(t, ig_prof / ig_prof.max(), label="IG")
cr = ex["cross"]; rows = [p - 1 for p in W["tok_pos"]]; te = (np.arange(n_a) + 0.5) * ENC_HOP
heads = RES[MAIN]["align"] or [(l, h) for l in range(cr.shape[0]) for h in range(cr.shape[1])]
ca = np.mean([cr[l, h][rows].mean(0) for l, h in heads], 0); ax[2].plot(te, ca / ca.max(), label="CrossAttn (Alignment-Köpfe)")
ax[2].axvspan(W["start"], W["end"], color="r", alpha=0.15); ax[2].legend(); ax[2].set_ylabel("norm. Relevanz")
A = np.mean([cr[l, h] for l, h in heads], 0)
ax[3].imshow(A, aspect="auto", origin="upper", extent=[0, n_a * ENC_HOP, A.shape[0], 0], cmap="Blues", vmax=np.quantile(A, 0.995))
ax[3].set_yticks(np.arange(A.shape[0]) + 0.5); ax[3].set_yticklabels(ex["tokens"][:A.shape[0]], fontsize=6)
mark_words(ax[3], P["ref_words"], label=False); ax[3].set_xlabel("Zeit [s]"); ax[3].set_title("Cross-Attention der Alignment-Köpfe (Zeile = Query, die das Token vorhersagt)")
plt.tight_layout(); plt.savefig(f"{OUT}/example.png", dpi=130); plt.show()

## 7c Konvergenz von Integrated Gradients

Nach dem Axiom der *Completeness* muss $\sum_i \mathrm{IG}_i = f(x)-f(x_0)$ gelten. Der relative Fehler zeigt, ob die Riemann-Näherung des Integrals ausreicht; Zielwert ist < 5 %. Im ersten Lauf mit 32 Stützstellen lag der Median bei 32 % (tiny) bzw. 28 % (base). Deshalb wird hier für `n_conv` Äußerungen der Fehler für 16 bis 256 Stützstellen gemessen. Die Hauptanalyse nutzt `ig_steps` = 256.

**Begründung der Schrittzahl (Vorprüfung auf denselben ersten 3 Äußerungen, je 45 Wörter):** Der Median-Fehler fiel von 32 % / 39 % (32 Schritte, tiny / base) über 7 % / 9 % (64) und 1,2 % / 1,9 % (128) auf 0,2 % / 0,2 % (256). Unter 5 % lagen bei 128 Schritten erst 84 % / 67 % der Wörter, bei 256 Schritten 100 % / 98 %. Der Fehler sinkt pro Verdopplung etwa um den Faktor 4–5, wie es für die Mittelpunktsregel bei glattem Integranden zu erwarten ist. Die fehlende Konvergenz im ersten Lauf lag also allein an zu wenigen Stützstellen.

In [ ]:
CV = pd.concat([RES[m]["ig_conv"] for m in MODELS])
conv = (CV.groupby(["model", "steps"]).rel_err
        .agg(Median="median", P90=lambda s: s.quantile(0.9), Anteil_unter_5pct=lambda s: (s < 0.05).mean())
        .round(3))
display(conv.unstack("model"))
fig, ax = plt.subplots(figsize=(7, 4))
for m, d in CV.groupby("model"):
    g = d.groupby("steps").rel_err
    ax.plot(g.median().index, g.median().values, marker="o", label=f"{m.split('/')[-1]}: Median")
    ax.fill_between(g.median().index, g.quantile(0.25).values, g.quantile(0.75).values, alpha=0.2, label=f"{m.split('/')[-1]}: IQR")
ax.axhline(0.05, color="k", ls=":", lw=0.8, label="Ziel 5 %"); ax.axvline(CFG["ig_steps"], color="r", ls="--", lw=0.8, label="ig_steps (Hauptlauf)")
ax.set_xscale("log", base=2); ax.set_yscale("log"); ax.set_xlabel("Stützstellen"); ax.set_ylabel("rel. Completeness-Fehler")
ax.set_title("IG-Konvergenz"); ax.legend(fontsize=7)
plt.tight_layout(); plt.savefig(f"{OUT}/ig_convergence.png", dpi=130); plt.show()

## 8 TF1 – Zeitliche Lokalisierung

**H1:** Gradientenbasierte Attribution konzentriert sich deutlich über Zufallsniveau auf das Wortsegment, mit Kontext von wenigen 100 ms.

Getestet wird einseitig per Wilcoxon-Vorzeichen-Rang-Test (`mass_in` > `chance`) auf **Äußerungsebene** (Mittel je Äußerung, n = 100), Holm-korrigiert über alle Modell-Methoden-Kombinationen. Vor- und Nachkontext werden ebenso gepaart verglichen. Für `mass_in` und Pointing werden 95-%-Cluster-Bootstrap-Intervalle angegeben. Ein Überhang in den *Nachkontext* wäre ein Hinweis auf Koartikulation bzw. darauf, dass der bidirektionale Encoder spätere Frames einbezieht.

In [ ]:
W_ALL = pd.concat([RES[m]["words"] for m in MODELS])
summary = (W_ALL.groupby(["model", "method"])
           .agg(n=("mass_in", "size"), mass_in=("mass_in", "mean"), chance=("chance", "mean"),
                lift_median=("lift", "median"), pointing=("hit", "mean"), pointing_chance=("hit_chance", "mean"),
                before=("before", "mean"), after=("after", "mean"),
                peak_off_ms=("peak_off", lambda s: 1000 * s.median()))
           .round(3))
display(summary)

# Tests auf Äußerungsebene + Cluster-Bootstrap-KI
U = W_ALL.groupby(["model", "method", "utt"])[["mass_in", "chance", "before", "after"]].mean().reset_index()
rows = []
for (m, meth), d in U.groupby(["model", "method"]):
    dw = W_ALL[(W_ALL.model == m) & (W_ALL.method == meth)]
    mi, pt = boot_ci(dw, "mass_in"), boot_ci(dw, "hit")
    rows.append(dict(model=m.split("/")[-1], method=meth, n_utt=len(d),
                     mass_in=dw.mass_in.mean(), mass_in_KI=f"[{mi[0]:.3f}, {mi[1]:.3f}]",
                     pointing=dw.hit.mean(), pointing_KI=f"[{pt[0]:.3f}, {pt[1]:.3f}]",
                     p_mass_in_gt_chance=stats.wilcoxon(d.mass_in - d.chance, alternative="greater").pvalue,
                     after_minus_before=(d.after - d.before).mean(),
                     p_after_vs_before=stats.wilcoxon(d.after - d.before).pvalue))
T1 = pd.DataFrame(rows)
T1["p_mass_in_holm"] = holm(T1.p_mass_in_gt_chance)
T1["p_after_before_holm"] = holm(T1.p_after_vs_before)
display(T1.drop(columns=["p_mass_in_gt_chance", "p_after_vs_before"]).set_index(["model", "method"])
        .style.format({"mass_in": "{:.3f}", "pointing": "{:.3f}", "after_minus_before": "{:+.3f}",
                       "p_mass_in_holm": "{:.1e}", "p_after_before_holm": "{:.1e}"}))

In [ ]:
centers = (BINS[:-1] + BINS[1:]) / 2
fig, ax = plt.subplots(len(MODELS), 2, figsize=(15, 4.3 * len(MODELS)), squeeze=False)
for i, m in enumerate(MODELS):
    E = RES[m]["erp"]
    for meth, d in E.groupby("method"):
        ax[i, 0].plot(centers, np.nanmean(np.stack(d.prof), 0), label=meth)
    for f, d in E[E.method == "IG"].groupby("func"):
        ax[i, 1].plot(centers, np.nanmean(np.stack(d.prof), 0), label="Funktionswort" if f else "Inhaltswort")
    for a, ttl in zip(ax[i], ["Alle Methoden", "IG: Funktions- vs. Inhaltswörter"]):
        a.axvspan(0, 1, color="tab:orange", alpha=0.15); a.axhline(1, color="k", ls=":", lw=0.8)
        a.set_xlabel("relative Zeit (0 = Wortanfang, 1 = Wortende)"); a.set_ylabel("Relevanz / Gleichverteilung")
        a.set_title(f"{m.split('/')[-1]}: {ttl}"); a.legend(fontsize=8)
plt.suptitle("Ereigniskorreliertes Relevanzprofil"); plt.tight_layout(); plt.savefig(f"{OUT}/erp.png", dpi=130); plt.show()

fig, ax = plt.subplots(len(MODELS), 3, figsize=(15, 3.8 * len(MODELS)), squeeze=False)
for i, m in enumerate(MODELS):
    d = W_ALL[(W_ALL.model == m) & (W_ALL.method == "IG")]
    d.boxplot("mass_in", by="func", ax=ax[i, 0]); ax[i, 0].set_title(f"{m.split('/')[-1]}: IG mass_in – Inhalt (False) vs. Funktion (True)")
    ax[i, 1].scatter(d.w_dur, d.mass_in, s=4, alpha=0.4); ax[i, 1].set_xlabel("Wortdauer [s]"); ax[i, 1].set_ylabel("IG mass_in")
    d.groupby("n_tok").hit.mean().plot.bar(ax=ax[i, 2]); ax[i, 2].set_title("Pointing-Accuracy nach Anzahl BPE-Tokens")
plt.suptitle(""); plt.tight_layout(); plt.savefig(f"{OUT}/tf1_wordtypes.png", dpi=130); plt.show()

# Funktions- vs. Inhaltswörter: roh und mit Kontrolle der Wortdauer
for m in MODELS:
    d = W_ALL[(W_ALL.model == m) & (W_ALL.method == "IG")].copy()
    d["Funktionswort"] = d.func.astype(float); d["log_Dauer"] = np.log(d.w_dur)
    d["log_lift"] = np.log(d.lift.clip(lower=1e-6)); d["Pointing_über_Zufall"] = d.hit - d.hit_chance
    mw = stats.mannwhitneyu(d[d.func].lift, d[~d.func].lift)
    rho = stats.spearmanr(d.w_dur, d.mass_in)
    print(f"{m}\n  roh: lift Funktions- vs. Inhaltswort Median {d[d.func].lift.median():.2f} vs {d[~d.func].lift.median():.2f} "
          f"(Mann-Whitney p={mw.pvalue:.1e}); Spearman(Wortdauer, mass_in) = {rho.statistic:.2f}")
    for y in ("log_lift", "Pointing_über_Zufall"):
        b = ols_boot(d, y, ["Funktionswort", "log_Dauer"]).loc["Funktionswort"]
        print(f"  {y} ~ Funktionswort + log(Dauer): β_Funktionswort = {b.coef:+.3f}  95-%-KI [{b.ci_lo:+.3f}, {b.ci_hi:+.3f}]")
    d["Dauer-Quintil"] = pd.qcut(d.w_dur, 5, labels=False, duplicates="drop") + 1
    strat = d.groupby(["Dauer-Quintil", "func"]).agg(lift=("lift", "median"), n=("lift", "size")).unstack("func")
    strat.columns = [f"{a} {'Funktionswort' if f else 'Inhaltswort'}" for a, f in strat.columns]
    display(strat.round(2))

### Interpretation TF1 – Zeitliche Lokalisierung (H1)

* **H1 bestätigt:** Für alle Methoden und beide Modelle liegt `mass_in` weit über `chance` (Wilcoxon p < 1e-287, `chance` ≈ 0,043). Der `lift` (Median) liegt zwischen 4,4 (Grad×Δ, base) und 18,6 (Occlusion, tiny). Das Pointing liegt für IG, Grad×Δ, Occlusion und die Alignment-Köpfe zwischen 0,63 und 0,92; der Mittelwert über alle Cross-Attention-Köpfe ist mit 0,03 / 0,05 die Ausnahme (Abschnitt 10).
* **Occlusion lokalisiert am schärfsten** (`mass_in` 0,64 / 0,57, Pointing 0,92 / 0,86 für tiny / base). IG ist diffuser (`mass_in` 0,23 / 0,21), trifft aber mit dem **Peak** trotzdem in 75 % / 74 % der Fälle das Wort. Das ist kein Widerspruch: Gradienten haben ein Hintergrundrauschen über 3000 Frames, das Maximum liegt aber im Wort. Hinzu kommt, dass die IG-Completeness nicht erreicht ist (Abschnitt 7b).
* **Kontext:** Bei IG, Grad×Δ und Occlusion ist die Masse vor und nach dem Wort etwa gleich groß (je ca. 0,09–0,11, Unterschied höchstens ±0,016). Es gibt also einen symmetrischen Kontext von wenigen 100 ms, aber **keinen Überhang in den Nachkontext**. Das spricht nicht für eine besondere Rolle späterer Frames, passt aber zu Koartikulation in beide Richtungen. Auffällig ist nur die Cross-Attention der Alignment-Köpfe von `whisper-tiny`: 0,222 mehr Masse nach als vor dem Wort (0,27 gegen 0,05, Peak-Offset +60 ms). Bei `whisper-base` fehlt dieser Versatz. Das ist eine Eigenschaft dieser Köpfe, kein allgemeiner Effekt, und bedeutet für Zeitstempel: Sie sind keine exakten Wortgrenzen.
* **Funktions- vs. Inhaltswörter:** Funktionswörter haben einen **höheren** `lift` (Median 7,13 gegen 5,62, p = 4e-18). Das widerspricht der naiven Erwartung schlechterer Lokalisierung. Der Grund ist wahrscheinlich das Zufallsniveau: Kurze Wörter haben eine kleine `chance`, so dass der `lift` leichter hoch wird. `mass_in` hängt stark an der Wortdauer (Spearman 0,70). Für Wortarten-Vergleiche ist `lift` daher nur eingeschränkt aussagekräftig.
* **Vorbehalt:** Alle IG-Werte sind wegen des Completeness-Fehlers (28–32 %) unsicher.

## 9 TF2 – Spektrale Merkmale

Für jedes Wort wird die |IG|-Masse im Wortsegment pro Mel-Bin normiert und mit dem Energieprofil $x-x_0$ im selben Segment verglichen. Ein Verhältnis > 1 heißt: Das Band ist *relevanter, als seine Energie erwarten lässt*.

**Achtung, Zirkularität:** IG $=(x-x_0)\odot\bar g$ enthält die Energie als Faktor. Eine Korrelation zwischen |IG| und Energie ist deshalb zum Teil ein Artefakt der Definition. Als energiefreie Vergleichsgröße dient der mittlere Gradient $|\bar g|$: Korreliert auch er mit der Energie, reagiert das Modell tatsächlich vor allem auf energiereiche Bänder.

Die Band-Occlusion liefert eine kausale Gegenprobe. Weil tiefe Bänder die meiste Energie tragen, ist ihr Entfernen der größte Eingriff. Deshalb wird der Abfall von $\log p$ zusätzlich **pro entfernter Energieeinheit** angegeben (Verhältnis der Mittelwerte über alle Wörter).

In [ ]:
freqs = librosa.mel_frequencies(n_mels=N_MEL + 2, fmin=0, fmax=SR / 2)[1:-1]   # Mittenfrequenzen der Slaney-Mel-Filter
fn = lambda f: "Funktionswort" if f else "Inhaltswort"
for m in MODELS:
    S, B, e = RES[m]["spec"], RES[m]["bands"], RES[m]["band_edges"]
    nb = len(e) - 1; bcols, ecols = [f"b{k}" for k in range(nb)], [f"e{k}" for k in range(nb)]
    bl = [f"{freqs[e[k]]:.0f}–{freqs[e[k + 1] - 1]:.0f}" for k in range(nb)]
    fig, ax = plt.subplots(1, 4, figsize=(21, 4))
    for f, d in S.groupby("func"):
        ig_m, g_m, en_m = np.stack(d.ig).mean(0), np.stack(d.grad).mean(0), np.stack(d.energy).mean(0)
        ax[0].plot(freqs, ig_m, label=f"|IG| {fn(f)}"); ax[0].plot(freqs, en_m, "--", label=f"Energie {fn(f)}")
        ax[1].plot(freqs, ig_m / en_m, label=f"|IG| / Energie, {fn(f)}"); ax[1].plot(freqs, g_m / en_m, ":", label=f"|ḡ| / Energie, {fn(f)}")
    ax[0].set_title("Anteil je Mel-Bin"); ax[1].set_title("Relevanzanteil / Energieanteil"); ax[1].axhline(1, color="k", ls=":")
    for a in ax[:2]: a.set_xscale("log"); a.set_xlabel("Hz"); a.legend(fontsize=7)
    g = B.groupby("func")
    drop = g[bcols].mean().T; drop.index = bl; drop.columns = [fn(c) for c in drop.columns]
    eff = pd.DataFrame(g[bcols].mean().to_numpy().T / g[ecols].mean().to_numpy().T, index=bl, columns=drop.columns)
    drop.plot.bar(ax=ax[2]); ax[2].set_title("Band-Occlusion: Abfall log p (Wortsegment)"); ax[2].set_xlabel("Hz")
    eff.plot.bar(ax=ax[3]); ax[3].set_title("Abfall log p je entfernter Energieeinheit Σ(x − x₀)"); ax[3].set_xlabel("Hz")
    plt.suptitle(m); plt.tight_layout(); plt.savefig(f"{OUT}/tf2_spectral_{m.split('/')[-1]}.png", dpi=130); plt.show()

    rho_ig = np.nanmedian([stats.spearmanr(a, b).statistic for a, b in zip(S.ig, S.energy)])
    rho_g = np.nanmedian([stats.spearmanr(a, b).statistic for a, b in zip(S.grad, S.energy)])
    all_drop = B[bcols].mean().to_numpy(); all_eff = all_drop / B[ecols].mean().to_numpy()
    print(f"{m}: Median Spearman je Wort – |IG| vs. Energie {rho_ig:.2f}, |ḡ| vs. Energie {rho_g:.2f}\n"
          f"  wichtigstes Band roh: {bl[all_drop.argmax()]} Hz | je Energieeinheit: {bl[all_eff.argmax()]} Hz")

### Interpretation TF2 – Spektrale Merkmale

* **Wichtigstes Band:** Die Band-Occlusion nennt **410–745 Hz** (Bereich des ersten Formanten) als wichtigstes Band. Laut Plot folgt der Bereich bis ca. 3,6 kHz, darüber trägt kaum Evidenz. Das passt zu dem, was für Sprachverständlichkeit erwartet wird.
* **IG zeichnet zum großen Teil Energie nach:** Die Rangkorrelation zwischen |IG|-Profil und Energieprofil im Wortsegment liegt im Median bei 0,67. IG ist also deutlich mit der Energie verbunden. Der Verhältnis-Plot (|IG|-Anteil / Energieanteil) zeigt, in welchen Bändern IG mehr oder weniger als die Energie erwarten lässt. Die kausale Gegenprobe (Band-Occlusion) ist deshalb der belastbarere Befund, weil sie nicht vom IG-Completeness-Problem betroffen ist.
* **Inhalts- vs. Funktionswörter:** Laut Plot reagieren Inhaltswörter in allen Bändern stärker als Funktionswörter.
* **Einschränkung:** Die Plots in Abschnitt 8 und 9 zeigen nur das erste Modell (`whisper-tiny`). Für `whisper-base` liegen sie nicht vor.

## 10 TF3 – Cross-Attention-Köpfe des Decoders

**H2:** Nur wenige Köpfe in mittleren und späten Decoder-Schichten richten ihre Cross-Attention eng an den Wortgrenzen aus. Diese sollten mit OpenAIs `alignment_heads` übereinstimmen.

Die Heatmap zeigt die Pointing-Accuracy pro (Schicht, Kopf). Die Alignment-Köpfe sind mit ★ markiert. Rechts steht der Anteil der Attention auf dem Padding.

In [ ]:
fig, axes = plt.subplots(len(MODELS), 2, figsize=(13, 3.2 * len(MODELS)), squeeze=False)
for i, m in enumerate(MODELS):
    r = RES[m]; chance = W_ALL[(W_ALL.model == m) & (W_ALL.method == "IG")].hit_chance.mean()
    for j, (k, ttl) in enumerate([("hit", f"Pointing-Accuracy (Zufall ≈ {chance:.2f})"), ("pad", "Masse auf Padding")]):
        a = axes[i, j]; M = r["cross"][k]
        im = a.imshow(M, cmap="viridis", vmin=0, vmax=1, aspect="auto"); plt.colorbar(im, ax=a)
        for (l, h), v in np.ndenumerate(M):
            a.text(h, l, f"{v:.2f}" + ("★" if r["align"] and (l, h) in r["align"] else ""), ha="center", va="center", fontsize=7, color="w")
        a.set_xlabel("Kopf"); a.set_ylabel("Decoder-Schicht"); a.set_title(f"{m.split('/')[-1]}: {ttl}")
plt.tight_layout(); plt.savefig(f"{OUT}/tf3_cross_heads.png", dpi=130); plt.show()

for m in MODELS:
    r = RES[m]; M = r["cross"]["hit"]; al = r["align"] or []; k = len(al) or 6
    top = [(int(l), int(h)) for l, h in zip(*np.unravel_index(np.argsort(-M, axis=None), M.shape))]
    print(f"{m}: Top-{k} Köpfe {top[:k]} | Überlappung mit alignment heads: {len(set(top[:k]) & set(al))}/{len(al)} | "
          f"Köpfe mit Pointing > 0.5: {(M > 0.5).sum()}/{M.size} | Mittel je Schicht: {M.mean(1).round(2)}")
    print("   Pointing der alignment heads:", {f"C{l}.{h}": round(float(M[l, h]), 2) for l, h in al})

### Interpretation TF3 – Cross-Attention-Köpfe (H2)

* **H2 teilweise bestätigt.** *Wenige Köpfe in späten Schichten* stimmt: Nur 7 von 24 (tiny) bzw. 9 von 48 (base) Köpfen haben ein Pointing > 0,5, fast alle in den späten Schichten (Mittel je Schicht tiny 0,03 → 0,65, base 0,02 → 0,58). Der beste Kopf erreicht in beiden Modellen 0,99.
* **Aber: Die Köpfe sind nur teilweise die `alignment_heads`.** Im Mittel sind sie deutlich besser als der Durchschnitt (Pointing 0,58 / 0,73 gegen 0,31 / 0,22). Unter den Top-6 bzw. Top-8 Köpfen sind aber nur 3 von 6 (tiny) bzw. 6 von 8 (base) Alignment-Köpfe. In `whisper-tiny` liegt der beste Kopf (Schicht 3 / Kopf 1) außerhalb der Alignment-Köpfe, und der schwächste Alignment-Kopf erreicht nur ca. 0,30. In `whisper-base` ist ein Alignment-Kopf (Schicht 5 / Kopf 2) laut Heatmap ein Padding-Sink (Pointing ≈ 0,04, Padding-Masse ≈ 0,88). Die Auswahl von OpenAI ist offenbar für Zeitstempel per DTW optimiert und nicht dasselbe wie „am besten lokalisierender Kopf“.
* **Mittelung über alle Köpfe ist unbrauchbar:** Pointing 0,03 / 0,05 (≈ Zufall), weil sich Sink-Köpfe und frühe Schichten mit den lokalisierenden Köpfen mischen. Es bleibt aber etwas Masse im Wort (`lift` 7,3 / 5,4).
* **Attention Sinks:** In den frühen Decoder-Schichten liegt ein großer Teil der Masse auf dem 30-s-Padding (bis ca. 0,87 / 0,88). Hohe Padding-Masse ist kein Lokalisierungsergebnis.

## 11 TF4 – Self-Attention-Köpfe des Encoders

**H3:** In frühen Encoder-Schichten dominieren lokale, diagonale Muster, in späteren Schichten globalere Muster.

**Tests.** (1) Spearman zwischen Schichtindex und Kopf-Kennzahl über alle Köpfe, einseitig in Richtung der Hypothese. Mit nur 4 bzw. 6 Schichten hat dieser Test wenig Teststärke. (2) Deshalb zusätzlich auf **Äußerungsebene**: Für jede Äußerung wird die Steigung des Schichtmittels von `local` bzw. `mean_dist` über die Schichten bestimmt und per Wilcoxon gegen 0 getestet (einseitig, n = 100). Dieser Test verallgemeinert über Eingaben, nicht über Modelle.

Heuristische Typisierung pro Kopf. Die Schwellen für `local` sind auf das Zufallsniveau bezogen: Bei einer mittleren Äußerung von ca. 340 Encoder-Frames liegen bei Gleichverteilung nur ca. 1,5 % der Masse im Fenster ±2 Frames.
* *Sink*: `pad_mass` > 0.5
* *Vorgänger/Nachfolger*: `prev` oder `next` > 0.3
* *lokal*: `local` ≥ 0.25 (mehr als das 15-Fache des Zufallsniveaus)
* *teils lokal*: `local` ≥ 0.10
* sonst *global*

In [ ]:
def head_type(s, l, h):
    if s["pad_mass"][l, h] > 0.5: return "Sink"
    if s["prev"][l, h] > 0.3: return "Vorgänger"
    if s["next"][l, h] > 0.3: return "Nachfolger"
    if s["local"][l, h] >= 0.25: return "lokal"
    if s["local"][l, h] >= 0.10: return "teils lokal"
    return "global"

keys = ["local", "mean_dist", "entropy", "pad_mass", "within_lift", "sil_lift"]
for m in MODELS:
    s, EU = RES[m]["enc"], RES[m]["enc_utt"]
    fig, ax = plt.subplots(1, len(keys), figsize=(3.2 * len(keys), 2.8))
    for a, k in zip(ax, keys):
        im = a.imshow(s[k], aspect="auto", cmap="magma"); plt.colorbar(im, ax=a); a.set_title(k)
        a.set_xlabel("Kopf"); a.set_ylabel("Encoder-Schicht")
    plt.suptitle(m); plt.tight_layout(); plt.savefig(f"{OUT}/tf4_enc_{m.split('/')[-1]}.png", dpi=130); plt.show()
    types = pd.DataFrame([[head_type(s, l, h) for h in range(s["local"].shape[1])] for l in range(s["local"].shape[0])])
    types.index.name = "Schicht"; display(types)

    L = np.repeat(np.arange(s["local"].shape[0]), s["local"].shape[1])
    r1 = stats.spearmanr(L, s["mean_dist"].ravel(), alternative="greater")
    r2 = stats.spearmanr(L, s["local"].ravel(), alternative="less")
    lay = np.arange(EU["local"].shape[1])
    sl_loc = np.array([np.polyfit(lay, u.mean(-1), 1)[0] for u in EU["local"]])
    sl_dist = np.array([np.polyfit(lay, u.mean(-1), 1)[0] for u in EU["mean_dist"]])
    p_loc = stats.wilcoxon(sl_loc, alternative="less").pvalue
    p_dist = stats.wilcoxon(sl_dist, alternative="greater").pvalue
    print(f"{m}\n  Köpfe:      Spearman(Schicht, mean_dist) = {r1.statistic:+.2f} (p einseitig = {r1.pvalue:.2g}), "
          f"Spearman(Schicht, local) = {r2.statistic:+.2f} (p einseitig = {r2.pvalue:.2g})")
    print(f"  Äußerungen: Steigung local je Schicht Median {np.median(sl_loc):+.4f} (Anteil < 0: {(sl_loc < 0).mean():.2f}, p = {p_loc:.2g}); "
          f"Steigung mean_dist Median {np.median(sl_dist):+.4f} s (Anteil > 0: {(sl_dist > 0).mean():.2f}, p = {p_dist:.2g})")
    display(pd.DataFrame({"local (Mittel)": EU["local"].mean(-1).mean(0), "local (SD über Köpfe)": s["local"].std(1),
                          "mean_dist [s]": EU["mean_dist"].mean(-1).mean(0),
                          "Typen": [", ".join(f"{t}×{n}" for t, n in types.loc[l].value_counts().items()) for l in types.index]},
                         index=pd.Index(lay, name="Schicht")).round(3))

ex = RES[MAIN]["example"]; Le, He = ex["enc"].shape[:2]
fig, ax = plt.subplots(Le, He, figsize=(2.1 * He, 2.1 * Le))
for l in range(Le):
    for h in range(He):
        a = ax[l, h]; M = ex["enc"][l, h].astype(np.float32)
        a.imshow(M, cmap="Greys", vmax=np.quantile(M, 0.995)); a.set_xticks([]); a.set_yticks([])
        a.set_title(f"L{l}H{h}: {head_type(RES[MAIN]['enc'], l, h)}", fontsize=7)
plt.suptitle(f"Encoder-Self-Attention (Audiobereich) – Beispiel, {MAIN}"); plt.tight_layout()
plt.savefig(f"{OUT}/tf4_enc_maps.png", dpi=110); plt.show()

### Interpretation TF4 – Encoder-Self-Attention (H3)

* **H3 ist statistisch nicht bestätigt.** Die Spearman-Korrelation über alle Köpfe liefert für `mean_dist` ρ = 0,22 (p = 0,30, tiny) und ρ = 0,12 (p = 0,41, base) und für `local` ρ = −0,20 (p = 0,34, tiny) und ρ = −0,26 (p = 0,077, base). Alle p-Werte liegen über 0,05, mit nur 24 bzw. 48 Köpfen ist die Teststärke allerdings gering. Die Richtung der Vorzeichen passt zur Hypothese (weiter und weniger lokal mit der Tiefe), ein belastbarer Trend ist es nicht.
* **Beschreibend** sinkt die mittlere Lokalität von Schicht 0 zur letzten Schicht (`local` tiny 0,41 → 0,16, base 0,35 → 0,12). Dieser Vergleich zweier Schichten ist nicht getestet und wird vor allem von wenigen sehr lokalen Köpfen in Schicht 0 getragen (`local` ≈ 0,95, `within_lift` ≈ 15), während einzelne Köpfe in späten Schichten weiter diagonal arbeiten. Der Übergang lokal → global gilt also höchstens im Mittel, nicht für jeden Kopf.
* **Die Heuristik zur Kopf-Typisierung ist zu streng:** Fast alle Köpfe landen in „global“, auch solche mit klarer Diagonale. Als Beschreibung ist die Kennzahlen-Karte besser als die Tabelle.
* **Attention Sinks** sind im Encoder selten (mittlere Padding-Masse 0,075 / 0,120). Auffällig ist in `whisper-base` Schicht 5 / Kopf 1 (laut Typisierung „Sink“).

## 12 TF5 – Faithfulness: Deletion und Ablation

**H4:** Attention-Gewichte sind schwächer kausal mit der Ausgabe verknüpft als Attribution. Konkret: (a) Die Deletion-Kurven von IG und Occlusion fallen steiler als die von Cross-Attention. (b) Die Lokalisierungsgüte eines Kopfes sagt seine Ablationswirkung nur schwach voraus.

**Vorsicht beim Deuten** (Hooker et al., 2019): Das Entfernen von Frames erzeugt Eingaben, die das Modell im Training nie gesehen hat. Die Silence-Baseline mildert das, beseitigt es aber nicht.

**Statistik:** Die Deletion-AUC wird je Wort als Mittel über die fünf Anteile berechnet. 95-%-KI per Cluster-Bootstrap über die 25 Äußerungen. Methodenvergleiche erfolgen gepaart (gleiche Wörter) mit Bootstrap-KI der Differenz und Wilcoxon auf Äußerungsebene.

In [ ]:
D = pd.concat([RES[m]["deletion"] for m in MODELS])
curves = D.groupby(["model", "method", "frac"]).delta.mean().unstack("frac")
display(curves.round(2))

AW = D.groupby(["model", "method", "utt", "wi"]).delta.mean().rename("auc").reset_index()   # AUC je Wort
rows = []
for (m, meth), d in AW.groupby(["model", "method"]):
    lo, hi = boot_ci(d, "auc")
    rows.append(dict(model=m.split("/")[-1], method=meth, AUC=d.auc.mean(), KI_unten=lo, KI_oben=hi))
auc = pd.DataFrame(rows).sort_values(["model", "AUC"]).set_index(["model", "method"]).round(2)
display(auc)

# gepaarte Vergleiche: negative Differenz = erste Methode fällt steiler (treuer)
PV = AW.pivot_table(index=["model", "utt", "wi"], columns="method", values="auc").reset_index()
pairs = [("CrossAttn (Alignment-Köpfe)", "IG"), ("CrossAttn (Alignment-Köpfe)", "Occlusion"),
         ("CrossAttn (alle Köpfe)", "IG"), ("Occlusion", "IG"), ("Grad×Δ", "IG")]
rows = []
for m, d in PV.groupby("model"):
    for a, b in pairs:
        if a not in d or b not in d: continue
        dd = d.assign(diff=d[a] - d[b]); lo, hi = boot_ci(dd, "diff"); du = dd.groupby("utt")["diff"].mean()
        rows.append(dict(model=m.split("/")[-1], Vergleich=f"{a} − {b}", Differenz=dd["diff"].mean(), KI_unten=lo, KI_oben=hi,
                         p_Wilcoxon=stats.wilcoxon(du).pvalue))
PAIR = pd.DataFrame(rows); PAIR["p_Holm"] = holm(PAIR.p_Wilcoxon)
display(PAIR.set_index(["model", "Vergleich"]).style.format({"Differenz": "{:+.2f}", "KI_unten": "{:+.2f}", "KI_oben": "{:+.2f}",
                                                              "p_Wilcoxon": "{:.2g}", "p_Holm": "{:.2g}"}))

fig, ax = plt.subplots(1, len(MODELS), figsize=(7 * len(MODELS), 4), squeeze=False)
for a, m in zip(ax[0], MODELS):
    for meth, d in D[D.model == m].groupby("method"):
        fr = sorted(d.frac.unique()); mu = [d[d.frac == f].delta.mean() for f in fr]
        ci = np.array([boot_ci(d[d.frac == f], "delta", n=500) for f in fr])
        ls = "--" if meth in ("Oracle", "Zufall") else "-"
        a.plot(fr, mu, marker="o", ls=ls, label=meth); a.fill_between(fr, ci[:, 0], ci[:, 1], alpha=0.12)
    a.set_xscale("log", base=2); a.set_xlabel("entfernte Frames (× Wortlänge)"); a.set_ylabel("Δ log p(Wort)"); a.set_title(m); a.legend(fontsize=7)
plt.tight_layout(); plt.savefig(f"{OUT}/tf5_deletion.png", dpi=130); plt.show()

In [ ]:
for m in MODELS:
    r = RES[m]; A = r["ablation"]
    fig, ax = plt.subplots(1, 3, figsize=(16, 3.6))
    for a, kind, L, H, ttl in [(ax[0], "enc_self", r["L_enc"], r["H_enc"], "Encoder-Self-Attn"),
                               (ax[1], "dec_cross", r["L_dec"], r["H_dec"], "Decoder-Cross-Attn")]:
        d = A[(A.kind == kind) & A.label.str.match(r"^[EC]\d")]
        M = np.zeros((L, H))
        for _, row in d.iterrows(): (l, h), = row.heads; M[l, h] = row.d_logprob
        im = a.imshow(M, cmap="Reds_r", aspect="auto"); plt.colorbar(im, ax=a)
        for (l, h), v in np.ndenumerate(M): a.text(h, l, f"{v:.2f}", ha="center", va="center", fontsize=7)
        a.set_title(f"{ttl}: Δ mittl. Token-log p bei Ablation"); a.set_xlabel("Kopf"); a.set_ylabel("Schicht")
    d = A[(A.kind == "dec_cross") & A.label.str.match(r"^C\d")]
    loc = np.array([r["cross"]["hit"][row.heads[0]] for _, row in d.iterrows()])
    ax[2].scatter(loc, d.d_logprob); ax[2].set_xlabel("Pointing-Accuracy des Kopfes"); ax[2].set_ylabel("Δ log p bei Ablation")
    rho = stats.spearmanr(loc, d.d_logprob)
    ax[2].set_title(f"Lokalisierung vs. kausale Wirkung (ρ={rho.statistic:.2f}, p={rho.pvalue:.2g})")
    plt.suptitle(m); plt.tight_layout(); plt.savefig(f"{OUT}/tf5_ablation_{m.split('/')[-1]}.png", dpi=130); plt.show()
    print(f"{m}: Spearman(Pointing, Δ log p) = {rho.statistic:+.2f}; Kendall τ = {stats.kendalltau(loc, d.d_logprob).statistic:+.2f}; "
          f"Köpfe mit |Δ log p| < 0,01: {(d.d_logprob.abs() < 0.01).sum()}/{len(d)}")

    # Gruppenablation: Zusammensetzung der Zufallsmengen und Bezug zur Einzelablation
    single = {row.heads[0]: row.d_logprob for _, row in d.iterrows()}
    top3 = sorted(single, key=single.get)[:3]
    g = A[~A.label.str.match(r"^[EC]\d")].copy()
    g["Köpfe"] = g.heads.apply(lambda hs: " ".join(f"C{l}.{h}" for l, h in hs))
    g["Top-3 darin"] = g.heads.apply(lambda hs: " ".join(f"C{l}.{h}" for l, h in hs if (l, h) in top3) or "–")
    g["Σ Einzel-Δ log p"] = g.heads.apply(lambda hs: sum(single[(l, h)] for l, h in hs))
    print(f"{m} – Gruppenablation (Top-3 der Einzelablation: {', '.join(f'C{l}.{h} ({single[(l, h)]:.2f})' for l, h in top3)})")
    cols = ["label", "Köpfe", "Top-3 darin", "Σ Einzel-Δ log p", "d_logprob"]
    if "wer" in g:
        cols += ["wer", "wer_capped", "d_wer_capped", "n_loop"]
    display(g[cols].round(3))
    rs = stats.spearmanr(g["Σ Einzel-Δ log p"], g.d_logprob)
    print(f"  Spearman(Σ Einzel-Δ log p, Gruppen-Δ log p) über {len(g)} Mengen = {rs.statistic:+.2f}")
    if "wer" in g:
        worst = g.loc[g.wer.idxmax()]; j = int(np.argmax(worst.wers))
        ref = r["utt"].set_index("utt").loc[worst.utt_ids[j], "ref_text"]
        print(f"  Beispiel ({worst.label}, WER {worst.wers[j]:.2f}):\n    REF: {ref[:200]}\n    HYP: {worst.texts[j].strip()[:300]}")

### Interpretation TF5 – Faithfulness (H4)

**Deletion-Test**

* Alle Methoden schlagen Zufall deutlich (Zufall bei 4 × Wortlänge nur −1,4 / −0,8, die Methoden −5 bis −6,5). Keine erreicht das Oracle (MFA-Wortmitte): Bei 1 × Wortlänge −6,0 / −5,0 für das Oracle gegen höchstens −4,5 / −3,1 für die Methoden.
* **H4a wird nicht bestätigt, eher widerlegt.** Die Cross-Attention der **Alignment-Köpfe** ist im Deletion-Test so treu wie IG: AUC tiny −3,42 gegen IG −3,65, base **−3,05 gegen IG −2,81 und Occlusion −2,93**, bei base also sogar etwas treuer als beide. Bei 2 × Wortlänge ist sie dort ebenfalls steiler (−5,52 gegen −4,52 bzw. −4,60). Nur der **Mittelwert über alle Köpfe** ist schwächer (−3,22 / −2,28). Die Hypothese „Attention ist schwächer kausal als Attribution“ gilt damit höchstens für ungefilterte Attention, nicht für gezielt ausgewählte Köpfe. Dabei ist zu bedenken, dass IG wegen der fehlenden Konvergenz geschwächt sein kann.
* Occlusion ist bei kleinen Anteilen (0,25–1 ×) am treuesten, was zu erwarten ist, weil sie genau dieselbe Größe misst, die der Test bewertet. Grad×Δ liegt knapp unter IG.
* Der **Out-of-Distribution-Effekt** (Hooker et al., 2019) bleibt: Das Setzen auf Stille erzeugt Eingaben, die Whisper nie gesehen hat.
* **Das Oracle ist nicht monoton:** Bei 4 × Wortlänge fällt log p weniger stark als bei 2 × (tiny −6,53 gegen −7,04, base −6,21 gegen −6,60). Mehr entfernte Information macht das Wort also wieder *wahrscheinlicher*. Informationsverlust allein kann das nicht erklären. Plausibel ist, dass große zusammenhängende Stille-Blöcke die Eingabe so weit aus der Trainingsverteilung schieben, dass das Modell stärker auf den Decoder-Kontext (Sprachmodell) zurückfällt und das per Teacher Forcing vorgegebene Wort wieder wahrscheinlicher wird. Das ist ein direkter Hinweis auf den OOD-Effekt und ein Grund, die Werte bei großen Anteilen vorsichtig zu lesen.

**Einzelkopf-Ablation**

* **H4b bestätigt:** Die kausal wichtigsten Cross-Attention-Köpfe lokalisieren schlecht (Pointing ≈ 0,08–0,11, z. B. tiny Schicht 1 / Kopf 2 mit Δ log p = −2,90 und base Schicht 2 / Kopf 2 mit −1,13). Köpfe mit hohem Pointing (> 0,5) ändern das Token-log p dagegen nur um ca. 0 bis −0,15. Lokalisierungsgüte sagt die Ablationswirkung nicht voraus.
* Das Spearman-ρ im Plot ist **negativ** (−0,56 / −0,54). Das ist als Rangeffekt zu lesen: Viele Köpfe ohne Lokalisierung haben eine Wirkung von praktisch genau 0, gut lokalisierende Köpfe kleine, aber von 0 verschiedene Effekte. Es bedeutet nicht, dass schlecht lokalisierende Köpfe im Mittel wichtiger wären.
* Im **Encoder** konzentriert sich die Wirkung auf Schicht 0. Bei `whisper-tiny` ist sie sehr groß (Kopf 4: −5,57, Kopf 2: −4,99), das sind die sehr lokalen, diagonalen Köpfe. Bei `whisper-base` ist der größte Einzeleffekt klein (−0,24), das Modell ist redundanter.

**Gruppenablation: Alignment-Köpfe gegen Zufallsmengen**

* Bei `whisper-tiny` schadet das Abschalten der Alignment-Köpfe (ΔWER +1,38) mehr als vier der fünf Zufallsmengen (+0,07 bis +0,37), aber weniger als Zufall 0 (+2,03). Bei `whisper-base` schaden **drei der fünf Zufallsmengen deutlich mehr** (+3,72, +4,67, +5,26) als die Alignment-Köpfe (+0,87). Es ist also **nicht** so, dass Alignment-Köpfe generell kausal wichtiger sind als gleich viele zufällige Köpfe. Das ist derselbe Befund wie bei der Einzelkopf-Ablation: Kausale Wichtigkeit und Zeitstempel-Lokalisierung sind verschiedene Eigenschaften.
* **Erklärung:** Einzelne Zufallsmengen enthalten Köpfe, die kausal sehr wichtig, aber nicht lokalisierend sind (z. B. Decoder Schicht 2 / Kopf 2 in base mit −1,13 oder Schicht 1 / Kopf 2 in tiny mit −2,90). Die Zusammensetzung der Zufallsmengen ist im Ergebnis gespeichert (Spalte `heads`) und wird oben zusammen mit der Summe der Einzelablationseffekte angezeigt.
* **WER über 100 %:** ΔWER-Werte über 1 und absolute WER von 1,46 bis 5,3 bedeuten, dass es mehr Fehler als Referenzwörter gibt. Das geht nur, wenn das Modell **mehr Wörter ausgibt als gesprochen wurden**, also halluziniert, sich wiederholt oder in eine Schleife läuft (Generierung bis `max_new_tokens = 200`). Die generierten Texte wurden nicht gespeichert, die Erklärung ist daher eine plausible Vermutung. Die Zahlen zeigen, dass die Ablation das Modell stark destabilisiert, und sind mit nur 25 Äußerungen und einer WER ohne Obergrenze ungenau. Sinnvoller wäre ein Vergleich über Δ log p oder eine auf 1 gekappte WER.
* **Stichprobe:** Deletion und Ablation nutzen nur 25 Äußerungen, ohne Konfidenzintervalle und ohne Wiederholungen mit anderen Seeds.

## 13 Modellvergleich tiny vs. base

In [ ]:
rows = []
for m in MODELS:
    r = RES[m]; w = r["words"]; A = r["ablation"]
    ig = w[w.method == "IG"]
    aw = r["deletion"].groupby(["method", "utt", "wi"]).delta.mean().rename("auc").reset_index()
    da = lambda meth: aw[aw.method == meth].auc.mean()
    row = dict(model=m, WER=r["utt"].wer.mean(), korrekte_Woerter=int(r["utt"].n_matched.sum()),
               IG_completeness_median=ig.ig_err.median(), IG_mass_in=ig.mass_in.mean(), IG_lift=ig.lift.median(),
               IG_pointing=ig.hit.mean(), Occlusion_pointing=w[w.method == "Occlusion"].hit.mean(),
               pointing_chance=ig.hit_chance.mean(),
               best_cross_head=r["cross"]["hit"].max(), mean_cross_head=r["cross"]["hit"].mean(),
               enc_local_first=r["enc"]["local"][0].mean(), enc_local_last=r["enc"]["local"][-1].mean(),
               enc_pad_mass=r["enc"]["pad_mass"].mean(),
               del_auc_IG=da("IG"), del_auc_Occlusion=da("Occlusion"), del_auc_CrossAttn_align=da("CrossAttn (Alignment-Köpfe)"),
               del_auc_Oracle=da("Oracle"), del_auc_Zufall=da("Zufall"))
    if r["align"]:
        row["align_heads_pointing"] = np.mean([r["cross"]["hit"][l, h] for l, h in r["align"]])
        if "d_wer_capped" in A:
            g = A[A.label == "alignment heads"].iloc[0]; z = A[A.label.str.startswith("Zufall")]
            row.update(dWERcap_align_heads=g.d_wer_capped, dWERcap_random_median=z.d_wer_capped.median(),
                       dWERcap_random_min=z.d_wer_capped.min(), dWERcap_random_max=z.d_wer_capped.max())
    rows.append(row)
CMP = pd.DataFrame(rows).set_index("model").T.astype(float).round(3); display(CMP)
CMP.to_csv(f"{OUT}/model_comparison.csv")
with open(f"{OUT}/summary.json", "w") as f:
    json.dump(dict(summary=summary.reset_index().to_dict("records"), comparison=CMP.to_dict(), cfg=CFG), f, indent=1, default=str)

## 14 Fazit – Beantwortung der Hypothesen

### Skalierung tiny vs. base (Abschnitt 13)

* `whisper-base` ist genauer (WER 0,066 gegen 0,084). Die IG-Attribution ist bei beiden ähnlich (Pointing 0,75 / 0,74), bei base aber etwas diffuser (`lift` 5,4 gegen 6,3).
* Die **Alignment-Köpfe lokalisieren in base deutlich besser** (Pointing 0,73 gegen 0,58, ohne den Nachlauf-Versatz von tiny). Gleichzeitig sind relativ weniger Köpfe lokalisierend (Mittel 0,22 gegen 0,31) und die Padding-Masse im Encoder ist höher (0,12 gegen 0,075).
* `whisper-base` ist **robuster** gegen Frame-Deletion (IG-AUC −2,81 gegen −3,65) und gegen die Ablation einzelner Encoder-Köpfe (max. −0,24 gegen −5,57). Das größere Modell ist redundanter.
* Beide Modelle sind klein, Aussagen über größere Whisper-Modelle lassen sich daraus nicht ableiten.

### Hypothesen

| | Hypothese | Ergebnis | Begründung |
|---|---|---|---|
| **H1** | Attribution liegt über Zufall im Wortsegment, mit Kontext von wenigen 100 ms. | **Bestätigt** (Vorbehalt bei IG) | `mass_in` > `chance` für alle Methoden (p < 1e-287), `lift` 4,4–18,6, symmetrischer Kontext (je ca. 0,09–0,11). IG nicht konvergiert (Fehler 28–32 %), Occlusion am schärfsten. |
| **H2** | Nur wenige Cross-Attention-Köpfe lokalisieren, und es sind die Alignment-Köpfe. | **Teilweise bestätigt** | 7/24 bzw. 9/48 Köpfe mit Pointing > 0,5, vor allem in späten Schichten. Alignment-Köpfe im Mittel besser (0,58 / 0,73 gegen 0,31 / 0,22), aber nicht die besten (Top-6/8-Überlappung 3/6 und 6/8), in base ist einer ein Sink. |
| **H3** | Frühe Encoder-Schichten sind lokal, späte global. | **Nicht bestätigt** | Spearman über die Schichten nicht signifikant (`mean_dist` p = 0,30 / 0,41, `local` p = 0,34 / 0,077). Nur beschreibend nimmt `local` von Schicht 0 zur letzten Schicht ab (0,41 → 0,16, 0,35 → 0,12). |
| **H4a** | Deletion-Kurven von IG und Occlusion fallen steiler als die der Cross-Attention. | **Nicht bestätigt** | Alignment-Köpfe etwa so treu wie IG (AUC −3,42 gegen −3,65 in tiny), in base sogar treuer (−3,05 gegen IG −2,81 und Occlusion −2,93). Nur der Mittelwert über alle Köpfe ist schwächer. |
| **H4b** | Lokalisierungsgüte eines Kopfes sagt seine Ablationswirkung nur schwach voraus. | **Bestätigt** | Die stärksten Ablationseffekte stammen von schlecht lokalisierenden Köpfen (Pointing ≈ 0,1). Gut lokalisierende Köpfe haben kaum Wirkung. Spearman ρ = −0,56 / −0,54. |

### Fazit

1. Whisper hört im Wesentlichen **im Wortsegment** hin, mit einem Kontext von wenigen 100 ms in beide Richtungen. Das zeigen alle Methoden, am schärfsten Occlusion.
2. Spektral zählt vor allem der Bereich um **410–745 Hz**.
3. **Attention taugt zur Lokalisierung nur in wenigen, ausgewählten Köpfen** der letzten Decoder-Schichten. Die `alignment_heads` sind dafür eine gute, aber keine optimale Wahl.
4. **Lokalisierung ist nicht Wichtigkeit.** Gut lokalisierende Köpfe sind kausal wenig wichtig, und die kausal wichtigsten Köpfe lokalisieren schlecht. Auch in der Gruppenablation sind die Alignment-Köpfe nicht generell wichtiger als Zufallsmengen.
5. Die Hypothese, Attention sei schwächer faithful als Attribution, gilt in den Daten nicht für gezielt gewählte Köpfe. Der Deletion-Test deckt aber nur eine Seite ab, die Ablation (H4b) zeigt die Grenze der Attention als Erklärung (Jain & Wallace, 2019).
6. Der erwartete Übergang lokal → global im Encoder lässt sich mit den vorhandenen Daten **nicht statistisch belegen**.

### Grenzen und Fehlerquellen

* **IG nicht konvergiert:** Completeness-Fehler im Median 32 % / 28 %, statt unter 5 %. Alle IG-Aussagen sind unsicher, ein Rerun mit größerem `ig_steps` (z. B. 128 oder mehr) steht aus.
* **Stichprobe:** 100 Äußerungen für Lokalisierung, nur 25 für Deletion und Ablation. Es gibt keine Konfidenzintervalle und keine Wiederholungen mit anderen Seeds. Die Gruppenablation (ΔWER) ist damit sehr ungenau.
* **WER über 100 %** in der Ablation deutet auf Halluzination oder Wiederholungen hin (Vermutung, Texte nicht gespeichert).
* **Out-of-Distribution-Effekt** (Hooker et al., 2019) beim Entfernen von Frames und beim Nullsetzen von Köpfen. Die Stille-Baseline mildert ihn nur.
* **Baseline-Abhängigkeit** von IG und Occlusion, **MFA-Ungenauigkeit** (ca. 10–20 ms bei 20 ms Toleranz), **Selektionsbias** (nur korrekt erkannte Wörter), **Teacher Forcing** auf die eigene Hypothese, Wörter aus mehreren Tokens.
* **Zufallsniveau und Wortdauer:** `mass_in` hängt von der Wortdauer ab (Spearman 0,70), der `lift` ist für Vergleiche von Wortarten nur eingeschränkt aussagekräftig.
* **Plots für TF1 und TF2** zeigen nur `whisper-tiny`.